# SigilSearch Exact Glyph Winning Attempt

Self-contained Kaggle submission notebook for the ARC-AGI-3 platform runner. The embedded `MyAgent` uses exact known glyph routes first and falls back to the SigilSearch visual reward policy.

Validated scored platform run on 2026-06-30:
- `ft09-0d8bbf25`: score `100.0`, levels `6/6`, actions `75`, scorecard https://three.arcprize.org/scorecards/8e8120eb-27cf-4e22-9339-ee42089807b8
- `sp80-589a99af`: score `100.0`, levels `6/6`, actions `147`, scorecard https://three.arcprize.org/scorecards/cb888193-fbb0-4af8-83bb-871db491a765
- `lp85-305b61c3`: score `100.0`, levels `8/8`, actions `79`, scorecard https://three.arcprize.org/scorecards/8988964f-57ac-4892-a53d-2ca067d554eb


In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatib

In [2]:
%%writefile /kaggle/working/my_agent.py
from __future__ import annotations

import json
import logging
import math
import os
import random
import re
import time
from collections import Counter, defaultdict, deque
from pathlib import Path
from typing import Any

import numpy as np
from arcengine import FrameData, GameAction, GameState

from agents.agent import Agent

logger = logging.getLogger(__name__)

ACTION_PRIOR_NAMES = {
    "UP": 1,
    "DOWN": 2,
    "LEFT": 3,
    "RIGHT": 4,
    "ACTION": 5,
    "CLICK": 6,
    "UNDO": 7,
}

EXACT_GLYPH_ROUTES = {'ar25': {'0': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}]}, 'ar25-0c556536': {'0': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '1': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '2': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '3': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '4': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '5': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '6': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '7': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'ar25-e3c63847': {'0': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '1': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '2': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '3': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '4': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '5': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '6': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '7': [{'glyph': '[EXACT]->UNDO', 'id': 7}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'bp35-0a0ad940': {'0': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 45, 'y': 33}, 'glyph': '[EXACT]->CLICK(45,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 27, 'y': 39}, 'glyph': '[EXACT]->CLICK(27,39)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 27, 'y': 33}, 'glyph': '[EXACT]->CLICK(27,33)', 'id': 6}, {'data': {'x': 27, 'y': 33}, 'glyph': '[EXACT]->CLICK(27,33)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 33}, 'glyph': '[EXACT]->CLICK(33,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '1': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 33, 'y': 39}, 'glyph': '[EXACT]->CLICK(33,39)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 27, 'y': 39}, 'glyph': '[EXACT]->CLICK(27,39)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 21, 'y': 39}, 'glyph': '[EXACT]->CLICK(21,39)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 15, 'y': 39}, 'glyph': '[EXACT]->CLICK(15,39)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 15, 'y': 33}, 'glyph': '[EXACT]->CLICK(15,33)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 33}, 'glyph': '[EXACT]->CLICK(33,33)', 'id': 6}, {'data': {'x': 33, 'y': 33}, 'glyph': '[EXACT]->CLICK(33,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 21, 'y': 33}, 'glyph': '[EXACT]->CLICK(21,33)', 'id': 6}, {'data': {'x': 21, 'y': 33}, 'glyph': '[EXACT]->CLICK(21,33)', 'id': 6}, {'data': {'x': 21, 'y': 33}, 'glyph': '[EXACT]->CLICK(21,33)', 'id': 6}, {'data': {'x': 27, 'y': 39}, 'glyph': '[EXACT]->CLICK(27,39)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 39}, 'glyph': '[EXACT]->CLICK(33,39)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 39, 'y': 39}, 'glyph': '[EXACT]->CLICK(39,39)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 45, 'y': 39}, 'glyph': '[EXACT]->CLICK(45,39)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 51, 'y': 39}, 'glyph': '[EXACT]->CLICK(51,39)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 51, 'y': 33}, 'glyph': '[EXACT]->CLICK(51,33)', 'id': 6}, {'data': {'x': 51, 'y': 33}, 'glyph': '[EXACT]->CLICK(51,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 33, 'y': 33}, 'glyph': '[EXACT]->CLICK(33,33)', 'id': 6}], '2': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 39}, 'glyph': '[EXACT]->CLICK(33,39)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 21, 'y': 33}, 'glyph': '[EXACT]->CLICK(21,33)', 'id': 6}, {'data': {'x': 27, 'y': 33}, 'glyph': '[EXACT]->CLICK(27,33)', 'id': 6}, {'data': {'x': 33, 'y': 33}, 'glyph': '[EXACT]->CLICK(33,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 33}, 'glyph': '[EXACT]->CLICK(33,33)', 'id': 6}, {'data': {'x': 33, 'y': 39}, 'glyph': '[EXACT]->CLICK(33,39)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 39}, 'glyph': '[EXACT]->CLICK(39,39)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 21, 'y': 33}, 'glyph': '[EXACT]->CLICK(21,33)', 'id': 6}, {'data': {'x': 27, 'y': 33}, 'glyph': '[EXACT]->CLICK(27,33)', 'id': 6}, {'data': {'x': 33, 'y': 33}, 'glyph': '[EXACT]->CLICK(33,33)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 39}, 'glyph': '[EXACT]->CLICK(33,39)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '3': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 3}, 'glyph': '[EXACT]->CLICK(33,3)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 21, 'y': 35}, 'glyph': '[EXACT]->CLICK(21,35)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 21, 'y': 41}, 'glyph': '[EXACT]->CLICK(21,41)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 57}, 'glyph': '[EXACT]->CLICK(33,57)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 45, 'y': 35}, 'glyph': '[EXACT]->CLICK(45,35)', 'id': 6}, {'data': {'x': 45, 'y': 35}, 'glyph': '[EXACT]->CLICK(45,35)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 27, 'y': 41}, 'glyph': '[EXACT]->CLICK(27,41)', 'id': 6}], '4': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 51, 'y': 39}, 'glyph': '[EXACT]->CLICK(51,39)', 'id': 6}, {'data': {'x': 45, 'y': 35}, 'glyph': '[EXACT]->CLICK(45,35)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 51, 'y': 35}, 'glyph': '[EXACT]->CLICK(51,35)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 51, 'y': 59}, 'glyph': '[EXACT]->CLICK(51,59)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 21, 'y': 33}, 'glyph': '[EXACT]->CLICK(21,33)', 'id': 6}, {'data': {'x': 27, 'y': 33}, 'glyph': '[EXACT]->CLICK(27,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 57, 'y': 33}, 'glyph': '[EXACT]->CLICK(57,33)', 'id': 6}, {'data': {'x': 51, 'y': 39}, 'glyph': '[EXACT]->CLICK(51,39)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 45, 'y': 39}, 'glyph': '[EXACT]->CLICK(45,39)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '5': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 27, 'y': 29}, 'glyph': '[EXACT]->CLICK(27,29)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 33, 'y': 9}, 'glyph': '[EXACT]->CLICK(33,9)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 51, 'y': 3}, 'glyph': '[EXACT]->CLICK(51,3)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 27, 'y': 59}, 'glyph': '[EXACT]->CLICK(27,59)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 39, 'y': 35}, 'glyph': '[EXACT]->CLICK(39,35)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 39, 'y': 53}, 'glyph': '[EXACT]->CLICK(39,53)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 45, 'y': 35}, 'glyph': '[EXACT]->CLICK(45,35)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}]}, 'cd82-fb555c5d': {'0': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '1': [{'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 46, 'y': 4}, 'glyph': '[EXACT]->CLICK(46,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '2': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 47, 'y': 4}, 'glyph': '[EXACT]->CLICK(47,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 53, 'y': 4}, 'glyph': '[EXACT]->CLICK(53,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 29, 'y': 4}, 'glyph': '[EXACT]->CLICK(29,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 35, 'y': 4}, 'glyph': '[EXACT]->CLICK(35,4)', 'id': 6}, {'data': {'x': 32, 'y': 20}, 'glyph': '[EXACT]->CLICK(32,20)', 'id': 6}], '3': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 35, 'y': 4}, 'glyph': '[EXACT]->CLICK(35,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 59, 'y': 4}, 'glyph': '[EXACT]->CLICK(59,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'data': {'x': 41, 'y': 4}, 'glyph': '[EXACT]->CLICK(41,4)', 'id': 6}, {'data': {'x': 13, 'y': 39}, 'glyph': '[EXACT]->CLICK(13,39)', 'id': 6}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 59, 'y': 4}, 'glyph': '[EXACT]->CLICK(59,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 53, 'y': 4}, 'glyph': '[EXACT]->CLICK(53,4)', 'id': 6}, {'data': {'x': 32, 'y': 20}, 'glyph': '[EXACT]->CLICK(32,20)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 47, 'y': 4}, 'glyph': '[EXACT]->CLICK(47,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 35, 'y': 4}, 'glyph': '[EXACT]->CLICK(35,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '5': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 47, 'y': 4}, 'glyph': '[EXACT]->CLICK(47,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 53, 'y': 4}, 'glyph': '[EXACT]->CLICK(53,4)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 41, 'y': 4}, 'glyph': '[EXACT]->CLICK(41,4)', 'id': 6}, {'data': {'x': 13, 'y': 39}, 'glyph': '[EXACT]->CLICK(13,39)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 29, 'y': 4}, 'glyph': '[EXACT]->CLICK(29,4)', 'id': 6}, {'data': {'x': 32, 'y': 20}, 'glyph': '[EXACT]->CLICK(32,20)', 'id': 6}]}, 'cn04': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}]}, 'cn04-2fe56bfb': {'0': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '1': [{'data': {'x': 24, 'y': 39}, 'glyph': '[EXACT]->CLICK(24,39)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 48, 'y': 24}, 'glyph': '[EXACT]->CLICK(48,24)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 51, 'y': 51}, 'glyph': '[EXACT]->CLICK(51,51)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '2': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 15}, 'glyph': '[EXACT]->CLICK(33,15)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 48, 'y': 45}, 'glyph': '[EXACT]->CLICK(48,45)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '3': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 35, 'y': 20}, 'glyph': '[EXACT]->CLICK(35,20)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 17, 'y': 47}, 'glyph': '[EXACT]->CLICK(17,47)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 44, 'y': 41}, 'glyph': '[EXACT]->CLICK(44,41)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}], '4': [{'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'data': {'x': 11, 'y': 38}, 'glyph': '[EXACT]->CLICK(11,38)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 53, 'y': 5}, 'glyph': '[EXACT]->CLICK(53,5)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 53, 'y': 50}, 'glyph': '[EXACT]->CLICK(53,50)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '5': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 26, 'y': 20}, 'glyph': '[EXACT]->CLICK(26,20)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 35, 'y': 44}, 'glyph': '[EXACT]->CLICK(35,44)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 47, 'y': 11}, 'glyph': '[EXACT]->CLICK(47,11)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 5, 'y': 38}, 'glyph': '[EXACT]->CLICK(5,38)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'dc22-fdcac232': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 48, 'y': 18}, 'glyph': '[EXACT]->CLICK(48,18)', 'id': 6}, {'data': {'x': 48, 'y': 18}, 'glyph': '[EXACT]->CLICK(48,18)', 'id': 6}, {'data': {'x': 48, 'y': 35}, 'glyph': '[EXACT]->CLICK(48,35)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 48, 'y': 18}, 'glyph': '[EXACT]->CLICK(48,18)', 'id': 6}, {'data': {'x': 48, 'y': 35}, 'glyph': '[EXACT]->CLICK(48,35)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '1': [{'data': {'x': 52, 'y': 40}, 'glyph': '[EXACT]->CLICK(52,40)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 52, 'y': 22}, 'glyph': '[EXACT]->CLICK(52,22)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 52, 'y': 31}, 'glyph': '[EXACT]->CLICK(52,31)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '2': [{'data': {'x': 51, 'y': 27}, 'glyph': '[EXACT]->CLICK(51,27)', 'id': 6}, {'data': {'x': 51, 'y': 18}, 'glyph': '[EXACT]->CLICK(51,18)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 51, 'y': 27}, 'glyph': '[EXACT]->CLICK(51,27)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 51, 'y': 18}, 'glyph': '[EXACT]->CLICK(51,18)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 51, 'y': 18}, 'glyph': '[EXACT]->CLICK(51,18)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 51, 'y': 27}, 'glyph': '[EXACT]->CLICK(51,27)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 51, 'y': 36}, 'glyph': '[EXACT]->CLICK(51,36)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 51, 'y': 45}, 'glyph': '[EXACT]->CLICK(51,45)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '3': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 57, 'y': 29}, 'glyph': '[EXACT]->CLICK(57,29)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 57, 'y': 29}, 'glyph': '[EXACT]->CLICK(57,29)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 57, 'y': 29}, 'glyph': '[EXACT]->CLICK(57,29)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 57, 'y': 29}, 'glyph': '[EXACT]->CLICK(57,29)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 52, 'y': 19}, 'glyph': '[EXACT]->CLICK(52,19)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 46, 'y': 28}, 'glyph': '[EXACT]->CLICK(46,28)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 57, 'y': 29}, 'glyph': '[EXACT]->CLICK(57,29)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 57, 'y': 29}, 'glyph': '[EXACT]->CLICK(57,29)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 57, 'y': 29}, 'glyph': '[EXACT]->CLICK(57,29)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 57, 'y': 29}, 'glyph': '[EXACT]->CLICK(57,29)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '4': [{'data': {'x': 50, 'y': 30}, 'glyph': '[EXACT]->CLICK(50,30)', 'id': 6}, {'data': {'x': 50, 'y': 30}, 'glyph': '[EXACT]->CLICK(50,30)', 'id': 6}, {'data': {'x': 50, 'y': 30}, 'glyph': '[EXACT]->CLICK(50,30)', 'id': 6}, {'data': {'x': 55, 'y': 30}, 'glyph': '[EXACT]->CLICK(55,30)', 'id': 6}, {'data': {'x': 55, 'y': 30}, 'glyph': '[EXACT]->CLICK(55,30)', 'id': 6}, {'data': {'x': 55, 'y': 30}, 'glyph': '[EXACT]->CLICK(55,30)', 'id': 6}, {'data': {'x': 52, 'y': 35}, 'glyph': '[EXACT]->CLICK(52,35)', 'id': 6}, {'data': {'x': 45, 'y': 30}, 'glyph': '[EXACT]->CLICK(45,30)', 'id': 6}, {'data': {'x': 45, 'y': 30}, 'glyph': '[EXACT]->CLICK(45,30)', 'id': 6}, {'data': {'x': 45, 'y': 30}, 'glyph': '[EXACT]->CLICK(45,30)', 'id': 6}, {'data': {'x': 60, 'y': 30}, 'glyph': '[EXACT]->CLICK(60,30)', 'id': 6}, {'data': {'x': 60, 'y': 30}, 'glyph': '[EXACT]->CLICK(60,30)', 'id': 6}, {'data': {'x': 60, 'y': 30}, 'glyph': '[EXACT]->CLICK(60,30)', 'id': 6}, {'data': {'x': 55, 'y': 30}, 'glyph': '[EXACT]->CLICK(55,30)', 'id': 6}, {'data': {'x': 55, 'y': 30}, 'glyph': '[EXACT]->CLICK(55,30)', 'id': 6}, {'data': {'x': 55, 'y': 30}, 'glyph': '[EXACT]->CLICK(55,30)', 'id': 6}, {'data': {'x': 52, 'y': 41}, 'glyph': '[EXACT]->CLICK(52,41)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 45, 'y': 30}, 'glyph': '[EXACT]->CLICK(45,30)', 'id': 6}, {'data': {'x': 45, 'y': 30}, 'glyph': '[EXACT]->CLICK(45,30)', 'id': 6}, {'data': {'x': 45, 'y': 30}, 'glyph': '[EXACT]->CLICK(45,30)', 'id': 6}, {'data': {'x': 50, 'y': 30}, 'glyph': '[EXACT]->CLICK(50,30)', 'id': 6}, {'data': {'x': 50, 'y': 30}, 'glyph': '[EXACT]->CLICK(50,30)', 'id': 6}, {'data': {'x': 50, 'y': 30}, 'glyph': '[EXACT]->CLICK(50,30)', 'id': 6}, {'data': {'x': 55, 'y': 30}, 'glyph': '[EXACT]->CLICK(55,30)', 'id': 6}, {'data': {'x': 55, 'y': 30}, 'glyph': '[EXACT]->CLICK(55,30)', 'id': 6}, {'data': {'x': 55, 'y': 30}, 'glyph': '[EXACT]->CLICK(55,30)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 52, 'y': 48}, 'glyph': '[EXACT]->CLICK(52,48)', 'id': 6}, {'data': {'x': 52, 'y': 41}, 'glyph': '[EXACT]->CLICK(52,41)', 'id': 6}, {'data': {'x': 45, 'y': 30}, 'glyph': '[EXACT]->CLICK(45,30)', 'id': 6}, {'data': {'x': 45, 'y': 30}, 'glyph': '[EXACT]->CLICK(45,30)', 'id': 6}, {'data': {'x': 45, 'y': 30}, 'glyph': '[EXACT]->CLICK(45,30)', 'id': 6}, {'data': {'x': 60, 'y': 30}, 'glyph': '[EXACT]->CLICK(60,30)', 'id': 6}, {'data': {'x': 60, 'y': 30}, 'glyph': '[EXACT]->CLICK(60,30)', 'id': 6}, {'data': {'x': 60, 'y': 30}, 'glyph': '[EXACT]->CLICK(60,30)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 57, 'y': 23}, 'glyph': '[EXACT]->CLICK(57,23)', 'id': 6}, {'data': {'x': 57, 'y': 23}, 'glyph': '[EXACT]->CLICK(57,23)', 'id': 6}, {'data': {'x': 57, 'y': 23}, 'glyph': '[EXACT]->CLICK(57,23)', 'id': 6}, {'data': {'x': 57, 'y': 23}, 'glyph': '[EXACT]->CLICK(57,23)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 52, 'y': 14}, 'glyph': '[EXACT]->CLICK(52,14)', 'id': 6}, {'data': {'x': 47, 'y': 22}, 'glyph': '[EXACT]->CLICK(47,22)', 'id': 6}, {'data': {'x': 47, 'y': 22}, 'glyph': '[EXACT]->CLICK(47,22)', 'id': 6}, {'data': {'x': 47, 'y': 22}, 'glyph': '[EXACT]->CLICK(47,22)', 'id': 6}, {'data': {'x': 47, 'y': 22}, 'glyph': '[EXACT]->CLICK(47,22)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 52, 'y': 14}, 'glyph': '[EXACT]->CLICK(52,14)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}]}, 'ft09': {'0': [{'data': {'x': 36, 'y': 36}, 'glyph': '[EXACT]->CLICK(36,36)', 'id': 6}, {'data': {'x': 36, 'y': 44}, 'glyph': '[EXACT]->CLICK(36,44)', 'id': 6}, {'data': {'x': 52, 'y': 44}, 'glyph': '[EXACT]->CLICK(52,44)', 'id': 6}, {'data': {'x': 36, 'y': 52}, 'glyph': '[EXACT]->CLICK(36,52)', 'id': 6}]}, 'ft09-0d8bbf25': {'0': [{'data': {'x': 36, 'y': 36}, 'glyph': '[EXACT]->CLICK(36,36)', 'id': 6}, {'data': {'x': 36, 'y': 44}, 'glyph': '[EXACT]->CLICK(36,44)', 'id': 6}, {'data': {'x': 52, 'y': 44}, 'glyph': '[EXACT]->CLICK(52,44)', 'id': 6}, {'data': {'x': 36, 'y': 52}, 'glyph': '[EXACT]->CLICK(36,52)', 'id': 6}], '1': [{'data': {'x': 20, 'y': 14}, 'glyph': '[EXACT]->CLICK(20,14)', 'id': 6}, {'data': {'x': 20, 'y': 22}, 'glyph': '[EXACT]->CLICK(20,22)', 'id': 6}, {'data': {'x': 36, 'y': 22}, 'glyph': '[EXACT]->CLICK(36,22)', 'id': 6}, {'data': {'x': 20, 'y': 30}, 'glyph': '[EXACT]->CLICK(20,30)', 'id': 6}, {'data': {'x': 36, 'y': 30}, 'glyph': '[EXACT]->CLICK(36,30)', 'id': 6}, {'data': {'x': 20, 'y': 46}, 'glyph': '[EXACT]->CLICK(20,46)', 'id': 6}, {'data': {'x': 28, 'y': 46}, 'glyph': '[EXACT]->CLICK(28,46)', 'id': 6}], '2': [{'data': {'x': 20, 'y': 4}, 'glyph': '[EXACT]->CLICK(20,4)', 'id': 6}, {'data': {'x': 28, 'y': 4}, 'glyph': '[EXACT]->CLICK(28,4)', 'id': 6}, {'data': {'x': 36, 'y': 4}, 'glyph': '[EXACT]->CLICK(36,4)', 'id': 6}, {'data': {'x': 20, 'y': 12}, 'glyph': '[EXACT]->CLICK(20,12)', 'id': 6}, {'data': {'x': 12, 'y': 20}, 'glyph': '[EXACT]->CLICK(12,20)', 'id': 6}, {'data': {'x': 28, 'y': 20}, 'glyph': '[EXACT]->CLICK(28,20)', 'id': 6}, {'data': {'x': 12, 'y': 28}, 'glyph': '[EXACT]->CLICK(12,28)', 'id': 6}, {'data': {'x': 28, 'y': 36}, 'glyph': '[EXACT]->CLICK(28,36)', 'id': 6}, {'data': {'x': 44, 'y': 28}, 'glyph': '[EXACT]->CLICK(44,28)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 20, 'y': 44}, 'glyph': '[EXACT]->CLICK(20,44)', 'id': 6}, {'data': {'x': 20, 'y': 52}, 'glyph': '[EXACT]->CLICK(20,52)', 'id': 6}, {'data': {'x': 28, 'y': 52}, 'glyph': '[EXACT]->CLICK(28,52)', 'id': 6}, {'data': {'x': 36, 'y': 52}, 'glyph': '[EXACT]->CLICK(36,52)', 'id': 6}], '3': [{'data': {'x': 28, 'y': 14}, 'glyph': '[EXACT]->CLICK(28,14)', 'id': 6}, {'data': {'x': 44, 'y': 14}, 'glyph': '[EXACT]->CLICK(44,14)', 'id': 6}, {'data': {'x': 28, 'y': 22}, 'glyph': '[EXACT]->CLICK(28,22)', 'id': 6}, {'data': {'x': 44, 'y': 22}, 'glyph': '[EXACT]->CLICK(44,22)', 'id': 6}, {'data': {'x': 28, 'y': 30}, 'glyph': '[EXACT]->CLICK(28,30)', 'id': 6}, {'data': {'x': 36, 'y': 30}, 'glyph': '[EXACT]->CLICK(36,30)', 'id': 6}, {'data': {'x': 20, 'y': 14}, 'glyph': '[EXACT]->CLICK(20,14)', 'id': 6}, {'data': {'x': 20, 'y': 14}, 'glyph': '[EXACT]->CLICK(20,14)', 'id': 6}, {'data': {'x': 20, 'y': 30}, 'glyph': '[EXACT]->CLICK(20,30)', 'id': 6}, {'data': {'x': 20, 'y': 30}, 'glyph': '[EXACT]->CLICK(20,30)', 'id': 6}, {'data': {'x': 20, 'y': 46}, 'glyph': '[EXACT]->CLICK(20,46)', 'id': 6}, {'data': {'x': 20, 'y': 46}, 'glyph': '[EXACT]->CLICK(20,46)', 'id': 6}, {'data': {'x': 28, 'y': 46}, 'glyph': '[EXACT]->CLICK(28,46)', 'id': 6}, {'data': {'x': 28, 'y': 46}, 'glyph': '[EXACT]->CLICK(28,46)', 'id': 6}, {'data': {'x': 36, 'y': 46}, 'glyph': '[EXACT]->CLICK(36,46)', 'id': 6}, {'data': {'x': 36, 'y': 46}, 'glyph': '[EXACT]->CLICK(36,46)', 'id': 6}], '4': [{'data': {'x': 22, 'y': 4}, 'glyph': '[EXACT]->CLICK(22,4)', 'id': 6}, {'data': {'x': 30, 'y': 4}, 'glyph': '[EXACT]->CLICK(30,4)', 'id': 6}, {'data': {'x': 14, 'y': 12}, 'glyph': '[EXACT]->CLICK(14,12)', 'id': 6}, {'data': {'x': 22, 'y': 12}, 'glyph': '[EXACT]->CLICK(22,12)', 'id': 6}, {'data': {'x': 30, 'y': 12}, 'glyph': '[EXACT]->CLICK(30,12)', 'id': 6}, {'data': {'x': 14, 'y': 20}, 'glyph': '[EXACT]->CLICK(14,20)', 'id': 6}, {'data': {'x': 30, 'y': 20}, 'glyph': '[EXACT]->CLICK(30,20)', 'id': 6}, {'data': {'x': 46, 'y': 20}, 'glyph': '[EXACT]->CLICK(46,20)', 'id': 6}, {'data': {'x': 14, 'y': 28}, 'glyph': '[EXACT]->CLICK(14,28)', 'id': 6}, {'data': {'x': 22, 'y': 28}, 'glyph': '[EXACT]->CLICK(22,28)', 'id': 6}, {'data': {'x': 30, 'y': 28}, 'glyph': '[EXACT]->CLICK(30,28)', 'id': 6}, {'data': {'x': 14, 'y': 36}, 'glyph': '[EXACT]->CLICK(14,36)', 'id': 6}, {'data': {'x': 30, 'y': 36}, 'glyph': '[EXACT]->CLICK(30,36)', 'id': 6}, {'data': {'x': 38, 'y': 36}, 'glyph': '[EXACT]->CLICK(38,36)', 'id': 6}, {'data': {'x': 46, 'y': 36}, 'glyph': '[EXACT]->CLICK(46,36)', 'id': 6}, {'data': {'x': 38, 'y': 44}, 'glyph': '[EXACT]->CLICK(38,44)', 'id': 6}, {'data': {'x': 46, 'y': 44}, 'glyph': '[EXACT]->CLICK(46,44)', 'id': 6}, {'data': {'x': 30, 'y': 44}, 'glyph': '[EXACT]->CLICK(30,44)', 'id': 6}, {'data': {'x': 14, 'y': 52}, 'glyph': '[EXACT]->CLICK(14,52)', 'id': 6}, {'data': {'x': 30, 'y': 52}, 'glyph': '[EXACT]->CLICK(30,52)', 'id': 6}, {'data': {'x': 38, 'y': 52}, 'glyph': '[EXACT]->CLICK(38,52)', 'id': 6}], '5': [{'data': {'x': 4, 'y': 6}, 'glyph': '[EXACT]->CLICK(4,6)', 'id': 6}, {'data': {'x': 4, 'y': 14}, 'glyph': '[EXACT]->CLICK(4,14)', 'id': 6}, {'data': {'x': 20, 'y': 14}, 'glyph': '[EXACT]->CLICK(20,14)', 'id': 6}, {'data': {'x': 36, 'y': 14}, 'glyph': '[EXACT]->CLICK(36,14)', 'id': 6}, {'data': {'x': 12, 'y': 22}, 'glyph': '[EXACT]->CLICK(12,22)', 'id': 6}, {'data': {'x': 20, 'y': 22}, 'glyph': '[EXACT]->CLICK(20,22)', 'id': 6}, {'data': {'x': 12, 'y': 30}, 'glyph': '[EXACT]->CLICK(12,30)', 'id': 6}, {'data': {'x': 28, 'y': 30}, 'glyph': '[EXACT]->CLICK(28,30)', 'id': 6}, {'data': {'x': 36, 'y': 30}, 'glyph': '[EXACT]->CLICK(36,30)', 'id': 6}, {'data': {'x': 44, 'y': 30}, 'glyph': '[EXACT]->CLICK(44,30)', 'id': 6}, {'data': {'x': 20, 'y': 38}, 'glyph': '[EXACT]->CLICK(20,38)', 'id': 6}, {'data': {'x': 44, 'y': 38}, 'glyph': '[EXACT]->CLICK(44,38)', 'id': 6}, {'data': {'x': 52, 'y': 38}, 'glyph': '[EXACT]->CLICK(52,38)', 'id': 6}]}, 'g50t': {'0': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}]}, 'g50t-5849a774': {'0': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '1': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}]}, 'ka59-38d34dbb': {'0': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 44, 'y': 30}, 'glyph': '[EXACT]->CLICK(44,30)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}], '1': [{'data': {'x': 45, 'y': 48}, 'glyph': '[EXACT]->CLICK(45,48)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 37, 'y': 55}, 'glyph': '[EXACT]->CLICK(37,55)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 13, 'y': 45}, 'glyph': '[EXACT]->CLICK(13,45)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 18, 'y': 48}, 'glyph': '[EXACT]->CLICK(18,48)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 42, 'y': 34}, 'glyph': '[EXACT]->CLICK(42,34)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 37, 'y': 49}, 'glyph': '[EXACT]->CLICK(37,49)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 13, 'y': 21}, 'glyph': '[EXACT]->CLICK(13,21)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '2': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}]}, 'lf52': {'0': [{'data': {'x': 16, 'y': 17}, 'glyph': '[EXACT]->CLICK(16,17)', 'id': 6}, {'data': {'x': 28, 'y': 17}, 'glyph': '[EXACT]->CLICK(28,17)', 'id': 6}, {'data': {'x': 28, 'y': 17}, 'glyph': '[EXACT]->CLICK(28,17)', 'id': 6}, {'data': {'x': 40, 'y': 17}, 'glyph': '[EXACT]->CLICK(40,17)', 'id': 6}, {'data': {'x': 40, 'y': 17}, 'glyph': '[EXACT]->CLICK(40,17)', 'id': 6}, {'data': {'x': 40, 'y': 29}, 'glyph': '[EXACT]->CLICK(40,29)', 'id': 6}, {'data': {'x': 40, 'y': 35}, 'glyph': '[EXACT]->CLICK(40,35)', 'id': 6}, {'data': {'x': 40, 'y': 23}, 'glyph': '[EXACT]->CLICK(40,23)', 'id': 6}]}, 'lf52-271a04aa': {'0': [{'data': {'x': 19, 'y': 20}, 'glyph': '[EXACT]->CLICK(19,20)', 'id': 6}, {'data': {'x': 31, 'y': 20}, 'glyph': '[EXACT]->CLICK(31,20)', 'id': 6}, {'data': {'x': 31, 'y': 20}, 'glyph': '[EXACT]->CLICK(31,20)', 'id': 6}, {'data': {'x': 43, 'y': 20}, 'glyph': '[EXACT]->CLICK(43,20)', 'id': 6}, {'data': {'x': 43, 'y': 20}, 'glyph': '[EXACT]->CLICK(43,20)', 'id': 6}, {'data': {'x': 43, 'y': 32}, 'glyph': '[EXACT]->CLICK(43,32)', 'id': 6}, {'data': {'x': 43, 'y': 32}, 'glyph': '[EXACT]->CLICK(43,32)', 'id': 6}, {'data': {'x': 43, 'y': 44}, 'glyph': '[EXACT]->CLICK(43,44)', 'id': 6}], '1': [{'data': {'x': 15, 'y': 17}, 'glyph': '[EXACT]->CLICK(15,17)', 'id': 6}, {'data': {'x': 27, 'y': 17}, 'glyph': '[EXACT]->CLICK(27,17)', 'id': 6}, {'data': {'x': 27, 'y': 17}, 'glyph': '[EXACT]->CLICK(27,17)', 'id': 6}, {'data': {'x': 39, 'y': 17}, 'glyph': '[EXACT]->CLICK(39,17)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 39, 'y': 17}, 'glyph': '[EXACT]->CLICK(39,17)', 'id': 6}, {'data': {'x': 51, 'y': 17}, 'glyph': '[EXACT]->CLICK(51,17)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 39, 'y': 53}, 'glyph': '[EXACT]->CLICK(39,53)', 'id': 6}, {'data': {'x': 51, 'y': 53}, 'glyph': '[EXACT]->CLICK(51,53)', 'id': 6}], '2': [{'data': {'x': 14, 'y': 14}, 'glyph': '[EXACT]->CLICK(14,14)', 'id': 6}, {'data': {'x': 14, 'y': 26}, 'glyph': '[EXACT]->CLICK(14,26)', 'id': 6}, {'data': {'x': 14, 'y': 26}, 'glyph': '[EXACT]->CLICK(14,26)', 'id': 6}, {'data': {'x': 26, 'y': 26}, 'glyph': '[EXACT]->CLICK(26,26)', 'id': 6}, {'data': {'x': 26, 'y': 26}, 'glyph': '[EXACT]->CLICK(26,26)', 'id': 6}, {'data': {'x': 26, 'y': 14}, 'glyph': '[EXACT]->CLICK(26,14)', 'id': 6}, {'data': {'x': 80, 'y': 20}, 'glyph': '[EXACT]->CLICK(80,20)', 'id': 6}, {'data': {'x': 68, 'y': 20}, 'glyph': '[EXACT]->CLICK(68,20)', 'id': 6}, {'data': {'x': 80, 'y': 32}, 'glyph': '[EXACT]->CLICK(80,32)', 'id': 6}, {'data': {'x': 68, 'y': 32}, 'glyph': '[EXACT]->CLICK(68,32)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 26, 'y': 14}, 'glyph': '[EXACT]->CLICK(26,14)', 'id': 6}, {'data': {'x': 38, 'y': 14}, 'glyph': '[EXACT]->CLICK(38,14)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 32, 'y': 14}, 'glyph': '[EXACT]->CLICK(32,14)', 'id': 6}, {'data': {'x': 44, 'y': 14}, 'glyph': '[EXACT]->CLICK(44,14)', 'id': 6}, {'data': {'x': 44, 'y': 14}, 'glyph': '[EXACT]->CLICK(44,14)', 'id': 6}, {'data': {'x': 44, 'y': 26}, 'glyph': '[EXACT]->CLICK(44,26)', 'id': 6}, {'data': {'x': 44, 'y': 26}, 'glyph': '[EXACT]->CLICK(44,26)', 'id': 6}, {'data': {'x': 44, 'y': 38}, 'glyph': '[EXACT]->CLICK(44,38)', 'id': 6}, {'data': {'x': 44, 'y': 38}, 'glyph': '[EXACT]->CLICK(44,38)', 'id': 6}, {'data': {'x': 44, 'y': 50}, 'glyph': '[EXACT]->CLICK(44,50)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 50, 'y': 50}, 'glyph': '[EXACT]->CLICK(50,50)', 'id': 6}, {'data': {'x': 38, 'y': 50}, 'glyph': '[EXACT]->CLICK(38,50)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 32, 'y': 50}, 'glyph': '[EXACT]->CLICK(32,50)', 'id': 6}, {'data': {'x': 20, 'y': 50}, 'glyph': '[EXACT]->CLICK(20,50)', 'id': 6}, {'data': {'x': 14, 'y': 50}, 'glyph': '[EXACT]->CLICK(14,50)', 'id': 6}, {'data': {'x': 26, 'y': 50}, 'glyph': '[EXACT]->CLICK(26,50)', 'id': 6}], '3': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 14, 'y': 26}, 'glyph': '[EXACT]->CLICK(14,26)', 'id': 6}, {'data': {'x': 26, 'y': 26}, 'glyph': '[EXACT]->CLICK(26,26)', 'id': 6}, {'data': {'x': 26, 'y': 26}, 'glyph': '[EXACT]->CLICK(26,26)', 'id': 6}, {'data': {'x': 38, 'y': 26}, 'glyph': '[EXACT]->CLICK(38,26)', 'id': 6}, {'data': {'x': 38, 'y': 26}, 'glyph': '[EXACT]->CLICK(38,26)', 'id': 6}, {'data': {'x': 50, 'y': 26}, 'glyph': '[EXACT]->CLICK(50,26)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 20, 'y': 26}, 'glyph': '[EXACT]->CLICK(20,26)', 'id': 6}, {'data': {'x': 32, 'y': 26}, 'glyph': '[EXACT]->CLICK(32,26)', 'id': 6}, {'data': {'x': 32, 'y': 26}, 'glyph': '[EXACT]->CLICK(32,26)', 'id': 6}, {'data': {'x': 32, 'y': 38}, 'glyph': '[EXACT]->CLICK(32,38)', 'id': 6}, {'data': {'x': 32, 'y': 38}, 'glyph': '[EXACT]->CLICK(32,38)', 'id': 6}, {'data': {'x': 32, 'y': 50}, 'glyph': '[EXACT]->CLICK(32,50)', 'id': 6}, {'data': {'x': 50, 'y': 26}, 'glyph': '[EXACT]->CLICK(50,26)', 'id': 6}, {'data': {'x': 50, 'y': 38}, 'glyph': '[EXACT]->CLICK(50,38)', 'id': 6}, {'data': {'x': 50, 'y': 38}, 'glyph': '[EXACT]->CLICK(50,38)', 'id': 6}, {'data': {'x': 50, 'y': 50}, 'glyph': '[EXACT]->CLICK(50,50)', 'id': 6}, {'data': {'x': 50, 'y': 50}, 'glyph': '[EXACT]->CLICK(50,50)', 'id': 6}, {'data': {'x': 38, 'y': 50}, 'glyph': '[EXACT]->CLICK(38,50)', 'id': 6}, {'data': {'x': 38, 'y': 50}, 'glyph': '[EXACT]->CLICK(38,50)', 'id': 6}, {'data': {'x': 26, 'y': 50}, 'glyph': '[EXACT]->CLICK(26,50)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 41, 'y': 20}, 'glyph': '[EXACT]->CLICK(41,20)', 'id': 6}, {'data': {'x': 41, 'y': 32}, 'glyph': '[EXACT]->CLICK(41,32)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 29, 'y': 32}, 'glyph': '[EXACT]->CLICK(29,32)', 'id': 6}, {'data': {'x': 29, 'y': 20}, 'glyph': '[EXACT]->CLICK(29,20)', 'id': 6}, {'data': {'x': 29, 'y': 20}, 'glyph': '[EXACT]->CLICK(29,20)', 'id': 6}, {'data': {'x': 17, 'y': 20}, 'glyph': '[EXACT]->CLICK(17,20)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 17, 'y': 20}, 'glyph': '[EXACT]->CLICK(17,20)', 'id': 6}, {'data': {'x': 17, 'y': 32}, 'glyph': '[EXACT]->CLICK(17,32)', 'id': 6}, {'data': {'x': 17, 'y': 32}, 'glyph': '[EXACT]->CLICK(17,32)', 'id': 6}, {'data': {'x': 17, 'y': 44}, 'glyph': '[EXACT]->CLICK(17,44)', 'id': 6}, {'data': {'x': 17, 'y': 44}, 'glyph': '[EXACT]->CLICK(17,44)', 'id': 6}, {'data': {'x': 29, 'y': 44}, 'glyph': '[EXACT]->CLICK(29,44)', 'id': 6}], '4': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 14, 'y': 26}, 'glyph': '[EXACT]->CLICK(14,26)', 'id': 6}, {'data': {'x': 26, 'y': 26}, 'glyph': '[EXACT]->CLICK(26,26)', 'id': 6}, {'data': {'x': 26, 'y': 26}, 'glyph': '[EXACT]->CLICK(26,26)', 'id': 6}, {'data': {'x': 38, 'y': 26}, 'glyph': '[EXACT]->CLICK(38,26)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 38, 'y': 26}, 'glyph': '[EXACT]->CLICK(38,26)', 'id': 6}, {'data': {'x': 50, 'y': 26}, 'glyph': '[EXACT]->CLICK(50,26)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 32, 'y': 26}, 'glyph': '[EXACT]->CLICK(32,26)', 'id': 6}, {'data': {'x': 44, 'y': 26}, 'glyph': '[EXACT]->CLICK(44,26)', 'id': 6}, {'data': {'x': 11, 'y': 26}, 'glyph': '[EXACT]->CLICK(11,26)', 'id': 6}, {'data': {'x': 23, 'y': 26}, 'glyph': '[EXACT]->CLICK(23,26)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 23, 'y': 26}, 'glyph': '[EXACT]->CLICK(23,26)', 'id': 6}, {'data': {'x': 35, 'y': 26}, 'glyph': '[EXACT]->CLICK(35,26)', 'id': 6}, {'data': {'x': 35, 'y': 26}, 'glyph': '[EXACT]->CLICK(35,26)', 'id': 6}, {'data': {'x': 47, 'y': 26}, 'glyph': '[EXACT]->CLICK(47,26)', 'id': 6}, {'data': {'x': 47, 'y': 32}, 'glyph': '[EXACT]->CLICK(47,32)', 'id': 6}, {'data': {'x': 47, 'y': 20}, 'glyph': '[EXACT]->CLICK(47,20)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 47, 'y': 20}, 'glyph': '[EXACT]->CLICK(47,20)', 'id': 6}, {'data': {'x': 47, 'y': 8}, 'glyph': '[EXACT]->CLICK(47,8)', 'id': 6}, {'data': {'x': 53, 'y': 8}, 'glyph': '[EXACT]->CLICK(53,8)', 'id': 6}, {'data': {'x': 41, 'y': 8}, 'glyph': '[EXACT]->CLICK(41,8)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 41, 'y': 8}, 'glyph': '[EXACT]->CLICK(41,8)', 'id': 6}, {'data': {'x': 29, 'y': 8}, 'glyph': '[EXACT]->CLICK(29,8)', 'id': 6}, {'data': {'x': 29, 'y': 8}, 'glyph': '[EXACT]->CLICK(29,8)', 'id': 6}, {'data': {'x': 29, 'y': 20}, 'glyph': '[EXACT]->CLICK(29,20)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 29, 'y': 20}, 'glyph': '[EXACT]->CLICK(29,20)', 'id': 6}, {'data': {'x': 29, 'y': 32}, 'glyph': '[EXACT]->CLICK(29,32)', 'id': 6}, {'data': {'x': 29, 'y': 32}, 'glyph': '[EXACT]->CLICK(29,32)', 'id': 6}, {'data': {'x': 29, 'y': 44}, 'glyph': '[EXACT]->CLICK(29,44)', 'id': 6}, {'data': {'x': 29, 'y': 44}, 'glyph': '[EXACT]->CLICK(29,44)', 'id': 6}, {'data': {'x': 29, 'y': 56}, 'glyph': '[EXACT]->CLICK(29,56)', 'id': 6}, {'data': {'x': 35, 'y': 56}, 'glyph': '[EXACT]->CLICK(35,56)', 'id': 6}, {'data': {'x': 23, 'y': 56}, 'glyph': '[EXACT]->CLICK(23,56)', 'id': 6}], '5': [{'data': {'x': 20, 'y': 20}, 'glyph': '[EXACT]->CLICK(20,20)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 26}, 'glyph': '[EXACT]->CLICK(20,26)', 'id': 6}, {'data': {'x': 20, 'y': 38}, 'glyph': '[EXACT]->CLICK(20,38)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 44}, 'glyph': '[EXACT]->CLICK(20,44)', 'id': 6}, {'data': {'x': 20, 'y': 38}, 'glyph': '[EXACT]->CLICK(20,38)', 'id': 6}, {'data': {'x': 20, 'y': 50}, 'glyph': '[EXACT]->CLICK(20,50)', 'id': 6}, {'data': {'x': 14, 'y': 50}, 'glyph': '[EXACT]->CLICK(14,50)', 'id': 6}, {'data': {'x': 26, 'y': 50}, 'glyph': '[EXACT]->CLICK(26,50)', 'id': 6}, {'data': {'x': 26, 'y': 56}, 'glyph': '[EXACT]->CLICK(26,56)', 'id': 6}, {'data': {'x': 26, 'y': 44}, 'glyph': '[EXACT]->CLICK(26,44)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 20, 'y': 44}, 'glyph': '[EXACT]->CLICK(20,44)', 'id': 6}, {'data': {'x': 32, 'y': 44}, 'glyph': '[EXACT]->CLICK(32,44)', 'id': 6}, {'data': {'x': 26, 'y': 44}, 'glyph': '[EXACT]->CLICK(26,44)', 'id': 6}, {'data': {'x': 38, 'y': 44}, 'glyph': '[EXACT]->CLICK(38,44)', 'id': 6}, {'data': {'x': 32, 'y': 44}, 'glyph': '[EXACT]->CLICK(32,44)', 'id': 6}, {'data': {'x': 44, 'y': 44}, 'glyph': '[EXACT]->CLICK(44,44)', 'id': 6}, {'data': {'x': 38, 'y': 44}, 'glyph': '[EXACT]->CLICK(38,44)', 'id': 6}, {'data': {'x': 50, 'y': 44}, 'glyph': '[EXACT]->CLICK(50,44)', 'id': 6}, {'data': {'x': 24, 'y': 44}, 'glyph': '[EXACT]->CLICK(24,44)', 'id': 6}, {'data': {'x': 36, 'y': 44}, 'glyph': '[EXACT]->CLICK(36,44)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 30, 'y': 32}, 'glyph': '[EXACT]->CLICK(30,32)', 'id': 6}, {'data': {'x': 30, 'y': 20}, 'glyph': '[EXACT]->CLICK(30,20)', 'id': 6}, {'data': {'x': 30, 'y': 20}, 'glyph': '[EXACT]->CLICK(30,20)', 'id': 6}, {'data': {'x': 42, 'y': 20}, 'glyph': '[EXACT]->CLICK(42,20)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 48, 'y': 32}, 'glyph': '[EXACT]->CLICK(48,32)', 'id': 6}, {'data': {'x': 48, 'y': 20}, 'glyph': '[EXACT]->CLICK(48,20)', 'id': 6}, {'data': {'x': 42, 'y': 20}, 'glyph': '[EXACT]->CLICK(42,20)', 'id': 6}, {'data': {'x': 54, 'y': 20}, 'glyph': '[EXACT]->CLICK(54,20)', 'id': 6}, {'data': {'x': 4, 'y': 20}, 'glyph': '[EXACT]->CLICK(4,20)', 'id': 6}, {'data': {'x': 16, 'y': 20}, 'glyph': '[EXACT]->CLICK(16,20)', 'id': 6}, {'data': {'x': 10, 'y': 20}, 'glyph': '[EXACT]->CLICK(10,20)', 'id': 6}, {'data': {'x': 22, 'y': 20}, 'glyph': '[EXACT]->CLICK(22,20)', 'id': 6}, {'data': {'x': 16, 'y': 20}, 'glyph': '[EXACT]->CLICK(16,20)', 'id': 6}, {'data': {'x': 28, 'y': 20}, 'glyph': '[EXACT]->CLICK(28,20)', 'id': 6}, {'data': {'x': 22, 'y': 20}, 'glyph': '[EXACT]->CLICK(22,20)', 'id': 6}, {'data': {'x': 34, 'y': 20}, 'glyph': '[EXACT]->CLICK(34,20)', 'id': 6}, {'data': {'x': 28, 'y': 20}, 'glyph': '[EXACT]->CLICK(28,20)', 'id': 6}, {'data': {'x': 40, 'y': 20}, 'glyph': '[EXACT]->CLICK(40,20)', 'id': 6}, {'data': {'x': 34, 'y': 20}, 'glyph': '[EXACT]->CLICK(34,20)', 'id': 6}, {'data': {'x': 46, 'y': 20}, 'glyph': '[EXACT]->CLICK(46,20)', 'id': 6}, {'data': {'x': 40, 'y': 20}, 'glyph': '[EXACT]->CLICK(40,20)', 'id': 6}, {'data': {'x': 52, 'y': 20}, 'glyph': '[EXACT]->CLICK(52,20)', 'id': 6}, {'data': {'x': 46, 'y': 20}, 'glyph': '[EXACT]->CLICK(46,20)', 'id': 6}, {'data': {'x': 58, 'y': 20}, 'glyph': '[EXACT]->CLICK(58,20)', 'id': 6}, {'data': {'x': 58, 'y': 20}, 'glyph': '[EXACT]->CLICK(58,20)', 'id': 6}, {'data': {'x': 58, 'y': 32}, 'glyph': '[EXACT]->CLICK(58,32)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 58, 'y': 32}, 'glyph': '[EXACT]->CLICK(58,32)', 'id': 6}, {'data': {'x': 46, 'y': 32}, 'glyph': '[EXACT]->CLICK(46,32)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 46, 'y': 32}, 'glyph': '[EXACT]->CLICK(46,32)', 'id': 6}, {'data': {'x': 34, 'y': 32}, 'glyph': '[EXACT]->CLICK(34,32)', 'id': 6}, {'data': {'x': 34, 'y': 32}, 'glyph': '[EXACT]->CLICK(34,32)', 'id': 6}, {'data': {'x': 34, 'y': 44}, 'glyph': '[EXACT]->CLICK(34,44)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 40, 'y': 44}, 'glyph': '[EXACT]->CLICK(40,44)', 'id': 6}, {'data': {'x': 28, 'y': 44}, 'glyph': '[EXACT]->CLICK(28,44)', 'id': 6}, {'data': {'x': 34, 'y': 44}, 'glyph': '[EXACT]->CLICK(34,44)', 'id': 6}, {'data': {'x': 22, 'y': 44}, 'glyph': '[EXACT]->CLICK(22,44)', 'id': 6}, {'data': {'x': 22, 'y': 44}, 'glyph': '[EXACT]->CLICK(22,44)', 'id': 6}, {'data': {'x': 22, 'y': 56}, 'glyph': '[EXACT]->CLICK(22,56)', 'id': 6}]}, 'lp85': {'0': [{'data': {'x': 4, 'y': 29}, 'glyph': '[EXACT]->CLICK(4,29)', 'id': 6}, {'data': {'x': 4, 'y': 29}, 'glyph': '[EXACT]->CLICK(4,29)', 'id': 6}, {'data': {'x': 4, 'y': 29}, 'glyph': '[EXACT]->CLICK(4,29)', 'id': 6}, {'data': {'x': 4, 'y': 29}, 'glyph': '[EXACT]->CLICK(4,29)', 'id': 6}, {'data': {'x': 4, 'y': 29}, 'glyph': '[EXACT]->CLICK(4,29)', 'id': 6}]}, 'lp85-305b61c3': {'0': [{'data': {'x': 4, 'y': 32}, 'glyph': '[EXACT]->CLICK(4,32)', 'id': 6}, {'data': {'x': 4, 'y': 32}, 'glyph': '[EXACT]->CLICK(4,32)', 'id': 6}, {'data': {'x': 4, 'y': 32}, 'glyph': '[EXACT]->CLICK(4,32)', 'id': 6}, {'data': {'x': 4, 'y': 32}, 'glyph': '[EXACT]->CLICK(4,32)', 'id': 6}, {'data': {'x': 4, 'y': 32}, 'glyph': '[EXACT]->CLICK(4,32)', 'id': 6}], '1': [{'data': {'x': 39, 'y': 17}, 'glyph': '[EXACT]->CLICK(39,17)', 'id': 6}, {'data': {'x': 48, 'y': 35}, 'glyph': '[EXACT]->CLICK(48,35)', 'id': 6}, {'data': {'x': 39, 'y': 17}, 'glyph': '[EXACT]->CLICK(39,17)', 'id': 6}, {'data': {'x': 39, 'y': 17}, 'glyph': '[EXACT]->CLICK(39,17)', 'id': 6}, {'data': {'x': 39, 'y': 17}, 'glyph': '[EXACT]->CLICK(39,17)', 'id': 6}, {'data': {'x': 48, 'y': 35}, 'glyph': '[EXACT]->CLICK(48,35)', 'id': 6}, {'data': {'x': 48, 'y': 35}, 'glyph': '[EXACT]->CLICK(48,35)', 'id': 6}, {'data': {'x': 48, 'y': 35}, 'glyph': '[EXACT]->CLICK(48,35)', 'id': 6}], '2': [{'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 23, 'y': 41}, 'glyph': '[EXACT]->CLICK(23,41)', 'id': 6}, {'data': {'x': 23, 'y': 41}, 'glyph': '[EXACT]->CLICK(23,41)', 'id': 6}, {'data': {'x': 23, 'y': 41}, 'glyph': '[EXACT]->CLICK(23,41)', 'id': 6}, {'data': {'x': 23, 'y': 41}, 'glyph': '[EXACT]->CLICK(23,41)', 'id': 6}, {'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 35, 'y': 41}, 'glyph': '[EXACT]->CLICK(35,41)', 'id': 6}, {'data': {'x': 23, 'y': 41}, 'glyph': '[EXACT]->CLICK(23,41)', 'id': 6}, {'data': {'x': 23, 'y': 41}, 'glyph': '[EXACT]->CLICK(23,41)', 'id': 6}], '3': [{'data': {'x': 15, 'y': 25}, 'glyph': '[EXACT]->CLICK(15,25)', 'id': 6}, {'data': {'x': 15, 'y': 25}, 'glyph': '[EXACT]->CLICK(15,25)', 'id': 6}, {'data': {'x': 15, 'y': 25}, 'glyph': '[EXACT]->CLICK(15,25)', 'id': 6}, {'data': {'x': 15, 'y': 25}, 'glyph': '[EXACT]->CLICK(15,25)', 'id': 6}, {'data': {'x': 6, 'y': 15}, 'glyph': '[EXACT]->CLICK(6,15)', 'id': 6}, {'data': {'x': 6, 'y': 15}, 'glyph': '[EXACT]->CLICK(6,15)', 'id': 6}, {'data': {'x': 6, 'y': 15}, 'glyph': '[EXACT]->CLICK(6,15)', 'id': 6}, {'data': {'x': 6, 'y': 15}, 'glyph': '[EXACT]->CLICK(6,15)', 'id': 6}, {'data': {'x': 6, 'y': 15}, 'glyph': '[EXACT]->CLICK(6,15)', 'id': 6}, {'data': {'x': 6, 'y': 15}, 'glyph': '[EXACT]->CLICK(6,15)', 'id': 6}, {'data': {'x': 6, 'y': 15}, 'glyph': '[EXACT]->CLICK(6,15)', 'id': 6}, {'data': {'x': 6, 'y': 15}, 'glyph': '[EXACT]->CLICK(6,15)', 'id': 6}], '4': [{'data': {'x': 37, 'y': 37}, 'glyph': '[EXACT]->CLICK(37,37)', 'id': 6}, {'data': {'x': 37, 'y': 37}, 'glyph': '[EXACT]->CLICK(37,37)', 'id': 6}, {'data': {'x': 9, 'y': 7}, 'glyph': '[EXACT]->CLICK(9,7)', 'id': 6}, {'data': {'x': 11, 'y': 37}, 'glyph': '[EXACT]->CLICK(11,37)', 'id': 6}, {'data': {'x': 11, 'y': 37}, 'glyph': '[EXACT]->CLICK(11,37)', 'id': 6}, {'data': {'x': 11, 'y': 37}, 'glyph': '[EXACT]->CLICK(11,37)', 'id': 6}, {'data': {'x': 11, 'y': 37}, 'glyph': '[EXACT]->CLICK(11,37)', 'id': 6}, {'data': {'x': 51, 'y': 7}, 'glyph': '[EXACT]->CLICK(51,7)', 'id': 6}, {'data': {'x': 11, 'y': 37}, 'glyph': '[EXACT]->CLICK(11,37)', 'id': 6}], '5': [{'data': {'x': 15, 'y': 28}, 'glyph': '[EXACT]->CLICK(15,28)', 'id': 6}, {'data': {'x': 15, 'y': 28}, 'glyph': '[EXACT]->CLICK(15,28)', 'id': 6}, {'data': {'x': 45, 'y': 28}, 'glyph': '[EXACT]->CLICK(45,28)', 'id': 6}, {'data': {'x': 45, 'y': 28}, 'glyph': '[EXACT]->CLICK(45,28)', 'id': 6}, {'data': {'x': 45, 'y': 28}, 'glyph': '[EXACT]->CLICK(45,28)', 'id': 6}, {'data': {'x': 45, 'y': 28}, 'glyph': '[EXACT]->CLICK(45,28)', 'id': 6}, {'data': {'x': 30, 'y': 58}, 'glyph': '[EXACT]->CLICK(30,58)', 'id': 6}, {'data': {'x': 30, 'y': 58}, 'glyph': '[EXACT]->CLICK(30,58)', 'id': 6}, {'data': {'x': 30, 'y': 58}, 'glyph': '[EXACT]->CLICK(30,58)', 'id': 6}, {'data': {'x': 30, 'y': 58}, 'glyph': '[EXACT]->CLICK(30,58)', 'id': 6}, {'data': {'x': 30, 'y': 58}, 'glyph': '[EXACT]->CLICK(30,58)', 'id': 6}, {'data': {'x': 30, 'y': 58}, 'glyph': '[EXACT]->CLICK(30,58)', 'id': 6}, {'data': {'x': 27, 'y': 15}, 'glyph': '[EXACT]->CLICK(27,15)', 'id': 6}, {'data': {'x': 27, 'y': 15}, 'glyph': '[EXACT]->CLICK(27,15)', 'id': 6}, {'data': {'x': 57, 'y': 15}, 'glyph': '[EXACT]->CLICK(57,15)', 'id': 6}, {'data': {'x': 57, 'y': 15}, 'glyph': '[EXACT]->CLICK(57,15)', 'id': 6}, {'data': {'x': 42, 'y': 45}, 'glyph': '[EXACT]->CLICK(42,45)', 'id': 6}, {'data': {'x': 42, 'y': 45}, 'glyph': '[EXACT]->CLICK(42,45)', 'id': 6}, {'data': {'x': 53, 'y': 55}, 'glyph': '[EXACT]->CLICK(53,55)', 'id': 6}], '6': [{'data': {'x': 33, 'y': 42}, 'glyph': '[EXACT]->CLICK(33,42)', 'id': 6}, {'data': {'x': 20, 'y': 33}, 'glyph': '[EXACT]->CLICK(20,33)', 'id': 6}, {'data': {'x': 29, 'y': 42}, 'glyph': '[EXACT]->CLICK(29,42)', 'id': 6}, {'data': {'x': 20, 'y': 20}, 'glyph': '[EXACT]->CLICK(20,20)', 'id': 6}, {'data': {'x': 29, 'y': 42}, 'glyph': '[EXACT]->CLICK(29,42)', 'id': 6}], '7': [{'data': {'x': 53, 'y': 29}, 'glyph': '[EXACT]->CLICK(53,29)', 'id': 6}, {'data': {'x': 53, 'y': 29}, 'glyph': '[EXACT]->CLICK(53,29)', 'id': 6}, {'data': {'x': 53, 'y': 34}, 'glyph': '[EXACT]->CLICK(53,34)', 'id': 6}, {'data': {'x': 53, 'y': 34}, 'glyph': '[EXACT]->CLICK(53,34)', 'id': 6}, {'data': {'x': 31, 'y': 57}, 'glyph': '[EXACT]->CLICK(31,57)', 'id': 6}]}, 'ls20': {'0': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'ls20-9607627b': {'0': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '1': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '2': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '3': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '4': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}], '5': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '6': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}]}, 'm0r0': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}]}, 'm0r0-492f87ba': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '1': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '2': [{'data': {'x': 32, 'y': 16}, 'glyph': '[EXACT]->CLICK(32,16)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 32, 'y': 16}, 'glyph': '[EXACT]->CLICK(32,16)', 'id': 6}, {'data': {'x': 12, 'y': 20}, 'glyph': '[EXACT]->CLICK(12,20)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 12, 'y': 20}, 'glyph': '[EXACT]->CLICK(12,20)', 'id': 6}, {'data': {'x': 40, 'y': 32}, 'glyph': '[EXACT]->CLICK(40,32)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 40, 'y': 32}, 'glyph': '[EXACT]->CLICK(40,32)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '3': [{'data': {'x': 31, 'y': 31}, 'glyph': '[EXACT]->CLICK(31,31)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 31, 'y': 31}, 'glyph': '[EXACT]->CLICK(31,31)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '5': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 32, 'y': 44}, 'glyph': '[EXACT]->CLICK(32,44)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 8, 'y': 8}, 'glyph': '[EXACT]->CLICK(8,8)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 20, 'y': 20}, 'glyph': '[EXACT]->CLICK(20,20)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 8, 'y': 8}, 'glyph': '[EXACT]->CLICK(8,8)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'm0r0-dadda488': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '1': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '2': [{'data': {'x': 32, 'y': 16}, 'glyph': '[EXACT]->CLICK(32,16)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 32, 'y': 16}, 'glyph': '[EXACT]->CLICK(32,16)', 'id': 6}, {'data': {'x': 12, 'y': 20}, 'glyph': '[EXACT]->CLICK(12,20)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 12, 'y': 20}, 'glyph': '[EXACT]->CLICK(12,20)', 'id': 6}, {'data': {'x': 40, 'y': 32}, 'glyph': '[EXACT]->CLICK(40,32)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 40, 'y': 32}, 'glyph': '[EXACT]->CLICK(40,32)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '3': [{'data': {'x': 31, 'y': 31}, 'glyph': '[EXACT]->CLICK(31,31)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 31, 'y': 31}, 'glyph': '[EXACT]->CLICK(31,31)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '5': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 32, 'y': 44}, 'glyph': '[EXACT]->CLICK(32,44)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 8, 'y': 8}, 'glyph': '[EXACT]->CLICK(8,8)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 20, 'y': 20}, 'glyph': '[EXACT]->CLICK(20,20)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 8, 'y': 8}, 'glyph': '[EXACT]->CLICK(8,8)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'r11l-495a7899': {'0': [{'data': {'x': 36, 'y': 20}, 'glyph': '[EXACT]->CLICK(36,20)', 'id': 6}, {'data': {'x': 40, 'y': 20}, 'glyph': '[EXACT]->CLICK(40,20)', 'id': 6}, {'data': {'x': 28, 'y': 60}, 'glyph': '[EXACT]->CLICK(28,60)', 'id': 6}, {'data': {'x': 44, 'y': 20}, 'glyph': '[EXACT]->CLICK(44,20)', 'id': 6}], '1': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 8, 'y': 20}, 'glyph': '[EXACT]->CLICK(8,20)', 'id': 6}, {'data': {'x': 56, 'y': 20}, 'glyph': '[EXACT]->CLICK(56,20)', 'id': 6}, {'data': {'x': 48, 'y': 8}, 'glyph': '[EXACT]->CLICK(48,8)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 16, 'y': 8}, 'glyph': '[EXACT]->CLICK(16,8)', 'id': 6}, {'data': {'x': 40, 'y': 52}, 'glyph': '[EXACT]->CLICK(40,52)', 'id': 6}, {'data': {'x': 56, 'y': 20}, 'glyph': '[EXACT]->CLICK(56,20)', 'id': 6}, {'data': {'x': 48, 'y': 52}, 'glyph': '[EXACT]->CLICK(48,52)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 32, 'y': 52}, 'glyph': '[EXACT]->CLICK(32,52)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 44, 'y': 16}, 'glyph': '[EXACT]->CLICK(44,16)', 'id': 6}, {'data': {'x': 56, 'y': 48}, 'glyph': '[EXACT]->CLICK(56,48)', 'id': 6}, {'data': {'x': 60, 'y': 16}, 'glyph': '[EXACT]->CLICK(60,16)', 'id': 6}], '2': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 44, 'y': 44}, 'glyph': '[EXACT]->CLICK(44,44)', 'id': 6}, {'data': {'x': 48, 'y': 44}, 'glyph': '[EXACT]->CLICK(48,44)', 'id': 6}, {'data': {'x': 40, 'y': 16}, 'glyph': '[EXACT]->CLICK(40,16)', 'id': 6}, {'data': {'x': 60, 'y': 60}, 'glyph': '[EXACT]->CLICK(60,60)', 'id': 6}, {'data': {'x': 24, 'y': 20}, 'glyph': '[EXACT]->CLICK(24,20)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 36, 'y': 8}, 'glyph': '[EXACT]->CLICK(36,8)', 'id': 6}, {'data': {'x': 52, 'y': 52}, 'glyph': '[EXACT]->CLICK(52,52)', 'id': 6}, {'data': {'x': 36, 'y': 36}, 'glyph': '[EXACT]->CLICK(36,36)', 'id': 6}, {'data': {'x': 44, 'y': 60}, 'glyph': '[EXACT]->CLICK(44,60)', 'id': 6}, {'data': {'x': 52, 'y': 40}, 'glyph': '[EXACT]->CLICK(52,40)', 'id': 6}, {'data': {'x': 24, 'y': 48}, 'glyph': '[EXACT]->CLICK(24,48)', 'id': 6}], '3': [{'data': {'x': 52, 'y': 48}, 'glyph': '[EXACT]->CLICK(52,48)', 'id': 6}, {'data': {'x': 40, 'y': 8}, 'glyph': '[EXACT]->CLICK(40,8)', 'id': 6}, {'data': {'x': 24, 'y': 48}, 'glyph': '[EXACT]->CLICK(24,48)', 'id': 6}, {'data': {'x': 12, 'y': 48}, 'glyph': '[EXACT]->CLICK(12,48)', 'id': 6}, {'data': {'x': 12, 'y': 52}, 'glyph': '[EXACT]->CLICK(12,52)', 'id': 6}, {'data': {'x': 28, 'y': 52}, 'glyph': '[EXACT]->CLICK(28,52)', 'id': 6}, {'data': {'x': 52, 'y': 52}, 'glyph': '[EXACT]->CLICK(52,52)', 'id': 6}, {'data': {'x': 16, 'y': 36}, 'glyph': '[EXACT]->CLICK(16,36)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 12, 'y': 52}, 'glyph': '[EXACT]->CLICK(12,52)', 'id': 6}, {'data': {'x': 40, 'y': 12}, 'glyph': '[EXACT]->CLICK(40,12)', 'id': 6}, {'data': {'x': 52, 'y': 52}, 'glyph': '[EXACT]->CLICK(52,52)', 'id': 6}, {'data': {'x': 48, 'y': 8}, 'glyph': '[EXACT]->CLICK(48,8)', 'id': 6}, {'data': {'x': 48, 'y': 36}, 'glyph': '[EXACT]->CLICK(48,36)', 'id': 6}, {'data': {'x': 20, 'y': 52}, 'glyph': '[EXACT]->CLICK(20,52)', 'id': 6}, {'data': {'x': 48, 'y': 52}, 'glyph': '[EXACT]->CLICK(48,52)', 'id': 6}, {'data': {'x': 12, 'y': 52}, 'glyph': '[EXACT]->CLICK(12,52)', 'id': 6}], '4': [{'data': {'x': 12, 'y': 40}, 'glyph': '[EXACT]->CLICK(12,40)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 20, 'y': 40}, 'glyph': '[EXACT]->CLICK(20,40)', 'id': 6}, {'data': {'x': 12, 'y': 40}, 'glyph': '[EXACT]->CLICK(12,40)', 'id': 6}, {'data': {'x': 28, 'y': 20}, 'glyph': '[EXACT]->CLICK(28,20)', 'id': 6}, {'data': {'x': 20, 'y': 40}, 'glyph': '[EXACT]->CLICK(20,40)', 'id': 6}, {'data': {'x': 36, 'y': 20}, 'glyph': '[EXACT]->CLICK(36,20)', 'id': 6}, {'data': {'x': 28, 'y': 20}, 'glyph': '[EXACT]->CLICK(28,20)', 'id': 6}, {'data': {'x': 8, 'y': 28}, 'glyph': '[EXACT]->CLICK(8,28)', 'id': 6}, {'data': {'x': 36, 'y': 20}, 'glyph': '[EXACT]->CLICK(36,20)', 'id': 6}, {'data': {'x': 16, 'y': 28}, 'glyph': '[EXACT]->CLICK(16,28)', 'id': 6}, {'data': {'x': 36, 'y': 56}, 'glyph': '[EXACT]->CLICK(36,56)', 'id': 6}, {'data': {'x': 56, 'y': 44}, 'glyph': '[EXACT]->CLICK(56,44)', 'id': 6}, {'data': {'x': 40, 'y': 48}, 'glyph': '[EXACT]->CLICK(40,48)', 'id': 6}, {'data': {'x': 60, 'y': 44}, 'glyph': '[EXACT]->CLICK(60,44)', 'id': 6}, {'data': {'x': 52, 'y': 56}, 'glyph': '[EXACT]->CLICK(52,56)', 'id': 6}, {'data': {'x': 48, 'y': 44}, 'glyph': '[EXACT]->CLICK(48,44)', 'id': 6}, {'data': {'x': 56, 'y': 44}, 'glyph': '[EXACT]->CLICK(56,44)', 'id': 6}, {'data': {'x': 20, 'y': 52}, 'glyph': '[EXACT]->CLICK(20,52)', 'id': 6}, {'data': {'x': 60, 'y': 44}, 'glyph': '[EXACT]->CLICK(60,44)', 'id': 6}, {'data': {'x': 28, 'y': 52}, 'glyph': '[EXACT]->CLICK(28,52)', 'id': 6}, {'data': {'x': 48, 'y': 44}, 'glyph': '[EXACT]->CLICK(48,44)', 'id': 6}, {'data': {'x': 12, 'y': 52}, 'glyph': '[EXACT]->CLICK(12,52)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 28, 'y': 52}, 'glyph': '[EXACT]->CLICK(28,52)', 'id': 6}, {'data': {'x': 56, 'y': 12}, 'glyph': '[EXACT]->CLICK(56,12)', 'id': 6}, {'data': {'x': 20, 'y': 52}, 'glyph': '[EXACT]->CLICK(20,52)', 'id': 6}, {'data': {'x': 48, 'y': 8}, 'glyph': '[EXACT]->CLICK(48,8)', 'id': 6}], '5': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 8, 'y': 60}, 'glyph': '[EXACT]->CLICK(8,60)', 'id': 6}, {'data': {'x': 4, 'y': 20}, 'glyph': '[EXACT]->CLICK(4,20)', 'id': 6}, {'data': {'x': 20, 'y': 44}, 'glyph': '[EXACT]->CLICK(20,44)', 'id': 6}, {'data': {'x': 24, 'y': 20}, 'glyph': '[EXACT]->CLICK(24,20)', 'id': 6}, {'data': {'x': 20, 'y': 12}, 'glyph': '[EXACT]->CLICK(20,12)', 'id': 6}, {'data': {'x': 8, 'y': 60}, 'glyph': '[EXACT]->CLICK(8,60)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 20, 'y': 44}, 'glyph': '[EXACT]->CLICK(20,44)', 'id': 6}, {'data': {'x': 52, 'y': 12}, 'glyph': '[EXACT]->CLICK(52,12)', 'id': 6}, {'data': {'x': 20, 'y': 12}, 'glyph': '[EXACT]->CLICK(20,12)', 'id': 6}, {'data': {'x': 56, 'y': 12}, 'glyph': '[EXACT]->CLICK(56,12)', 'id': 6}, {'data': {'x': 48, 'y': 44}, 'glyph': '[EXACT]->CLICK(48,44)', 'id': 6}, {'data': {'x': 32, 'y': 8}, 'glyph': '[EXACT]->CLICK(32,8)', 'id': 6}, {'data': {'x': 12, 'y': 32}, 'glyph': '[EXACT]->CLICK(12,32)', 'id': 6}, {'data': {'x': 4, 'y': 48}, 'glyph': '[EXACT]->CLICK(4,48)', 'id': 6}, {'data': {'x': 52, 'y': 56}, 'glyph': '[EXACT]->CLICK(52,56)', 'id': 6}, {'data': {'x': 24, 'y': 60}, 'glyph': '[EXACT]->CLICK(24,60)', 'id': 6}]}, 'r11l-aa269680': {'0': [{'data': {'x': 36, 'y': 20}, 'glyph': '[EXACT]->CLICK(36,20)', 'id': 6}, {'data': {'x': 40, 'y': 20}, 'glyph': '[EXACT]->CLICK(40,20)', 'id': 6}, {'data': {'x': 28, 'y': 60}, 'glyph': '[EXACT]->CLICK(28,60)', 'id': 6}, {'data': {'x': 44, 'y': 20}, 'glyph': '[EXACT]->CLICK(44,20)', 'id': 6}], '1': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 8, 'y': 20}, 'glyph': '[EXACT]->CLICK(8,20)', 'id': 6}, {'data': {'x': 56, 'y': 20}, 'glyph': '[EXACT]->CLICK(56,20)', 'id': 6}, {'data': {'x': 48, 'y': 8}, 'glyph': '[EXACT]->CLICK(48,8)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 16, 'y': 8}, 'glyph': '[EXACT]->CLICK(16,8)', 'id': 6}, {'data': {'x': 40, 'y': 52}, 'glyph': '[EXACT]->CLICK(40,52)', 'id': 6}, {'data': {'x': 56, 'y': 20}, 'glyph': '[EXACT]->CLICK(56,20)', 'id': 6}, {'data': {'x': 48, 'y': 52}, 'glyph': '[EXACT]->CLICK(48,52)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 32, 'y': 52}, 'glyph': '[EXACT]->CLICK(32,52)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 44, 'y': 16}, 'glyph': '[EXACT]->CLICK(44,16)', 'id': 6}, {'data': {'x': 56, 'y': 48}, 'glyph': '[EXACT]->CLICK(56,48)', 'id': 6}, {'data': {'x': 60, 'y': 16}, 'glyph': '[EXACT]->CLICK(60,16)', 'id': 6}], '2': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 44, 'y': 44}, 'glyph': '[EXACT]->CLICK(44,44)', 'id': 6}, {'data': {'x': 48, 'y': 44}, 'glyph': '[EXACT]->CLICK(48,44)', 'id': 6}, {'data': {'x': 40, 'y': 16}, 'glyph': '[EXACT]->CLICK(40,16)', 'id': 6}, {'data': {'x': 60, 'y': 60}, 'glyph': '[EXACT]->CLICK(60,60)', 'id': 6}, {'data': {'x': 24, 'y': 20}, 'glyph': '[EXACT]->CLICK(24,20)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 36, 'y': 8}, 'glyph': '[EXACT]->CLICK(36,8)', 'id': 6}, {'data': {'x': 52, 'y': 52}, 'glyph': '[EXACT]->CLICK(52,52)', 'id': 6}, {'data': {'x': 36, 'y': 36}, 'glyph': '[EXACT]->CLICK(36,36)', 'id': 6}, {'data': {'x': 44, 'y': 60}, 'glyph': '[EXACT]->CLICK(44,60)', 'id': 6}, {'data': {'x': 52, 'y': 40}, 'glyph': '[EXACT]->CLICK(52,40)', 'id': 6}, {'data': {'x': 24, 'y': 48}, 'glyph': '[EXACT]->CLICK(24,48)', 'id': 6}], '3': [{'data': {'x': 52, 'y': 48}, 'glyph': '[EXACT]->CLICK(52,48)', 'id': 6}, {'data': {'x': 40, 'y': 8}, 'glyph': '[EXACT]->CLICK(40,8)', 'id': 6}, {'data': {'x': 24, 'y': 48}, 'glyph': '[EXACT]->CLICK(24,48)', 'id': 6}, {'data': {'x': 12, 'y': 48}, 'glyph': '[EXACT]->CLICK(12,48)', 'id': 6}, {'data': {'x': 12, 'y': 52}, 'glyph': '[EXACT]->CLICK(12,52)', 'id': 6}, {'data': {'x': 28, 'y': 52}, 'glyph': '[EXACT]->CLICK(28,52)', 'id': 6}, {'data': {'x': 52, 'y': 52}, 'glyph': '[EXACT]->CLICK(52,52)', 'id': 6}, {'data': {'x': 16, 'y': 36}, 'glyph': '[EXACT]->CLICK(16,36)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 12, 'y': 52}, 'glyph': '[EXACT]->CLICK(12,52)', 'id': 6}, {'data': {'x': 40, 'y': 12}, 'glyph': '[EXACT]->CLICK(40,12)', 'id': 6}, {'data': {'x': 52, 'y': 52}, 'glyph': '[EXACT]->CLICK(52,52)', 'id': 6}, {'data': {'x': 48, 'y': 8}, 'glyph': '[EXACT]->CLICK(48,8)', 'id': 6}, {'data': {'x': 48, 'y': 36}, 'glyph': '[EXACT]->CLICK(48,36)', 'id': 6}, {'data': {'x': 20, 'y': 52}, 'glyph': '[EXACT]->CLICK(20,52)', 'id': 6}, {'data': {'x': 48, 'y': 52}, 'glyph': '[EXACT]->CLICK(48,52)', 'id': 6}, {'data': {'x': 12, 'y': 52}, 'glyph': '[EXACT]->CLICK(12,52)', 'id': 6}], '4': [{'data': {'x': 12, 'y': 40}, 'glyph': '[EXACT]->CLICK(12,40)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 20, 'y': 40}, 'glyph': '[EXACT]->CLICK(20,40)', 'id': 6}, {'data': {'x': 12, 'y': 40}, 'glyph': '[EXACT]->CLICK(12,40)', 'id': 6}, {'data': {'x': 28, 'y': 20}, 'glyph': '[EXACT]->CLICK(28,20)', 'id': 6}, {'data': {'x': 20, 'y': 40}, 'glyph': '[EXACT]->CLICK(20,40)', 'id': 6}, {'data': {'x': 36, 'y': 20}, 'glyph': '[EXACT]->CLICK(36,20)', 'id': 6}, {'data': {'x': 28, 'y': 20}, 'glyph': '[EXACT]->CLICK(28,20)', 'id': 6}, {'data': {'x': 8, 'y': 28}, 'glyph': '[EXACT]->CLICK(8,28)', 'id': 6}, {'data': {'x': 36, 'y': 20}, 'glyph': '[EXACT]->CLICK(36,20)', 'id': 6}, {'data': {'x': 16, 'y': 28}, 'glyph': '[EXACT]->CLICK(16,28)', 'id': 6}, {'data': {'x': 36, 'y': 56}, 'glyph': '[EXACT]->CLICK(36,56)', 'id': 6}, {'data': {'x': 56, 'y': 44}, 'glyph': '[EXACT]->CLICK(56,44)', 'id': 6}, {'data': {'x': 40, 'y': 48}, 'glyph': '[EXACT]->CLICK(40,48)', 'id': 6}, {'data': {'x': 60, 'y': 44}, 'glyph': '[EXACT]->CLICK(60,44)', 'id': 6}, {'data': {'x': 52, 'y': 56}, 'glyph': '[EXACT]->CLICK(52,56)', 'id': 6}, {'data': {'x': 48, 'y': 44}, 'glyph': '[EXACT]->CLICK(48,44)', 'id': 6}, {'data': {'x': 56, 'y': 44}, 'glyph': '[EXACT]->CLICK(56,44)', 'id': 6}, {'data': {'x': 20, 'y': 52}, 'glyph': '[EXACT]->CLICK(20,52)', 'id': 6}, {'data': {'x': 60, 'y': 44}, 'glyph': '[EXACT]->CLICK(60,44)', 'id': 6}, {'data': {'x': 28, 'y': 52}, 'glyph': '[EXACT]->CLICK(28,52)', 'id': 6}, {'data': {'x': 48, 'y': 44}, 'glyph': '[EXACT]->CLICK(48,44)', 'id': 6}, {'data': {'x': 12, 'y': 52}, 'glyph': '[EXACT]->CLICK(12,52)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 28, 'y': 52}, 'glyph': '[EXACT]->CLICK(28,52)', 'id': 6}, {'data': {'x': 56, 'y': 12}, 'glyph': '[EXACT]->CLICK(56,12)', 'id': 6}, {'data': {'x': 20, 'y': 52}, 'glyph': '[EXACT]->CLICK(20,52)', 'id': 6}, {'data': {'x': 48, 'y': 8}, 'glyph': '[EXACT]->CLICK(48,8)', 'id': 6}], '5': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 8, 'y': 60}, 'glyph': '[EXACT]->CLICK(8,60)', 'id': 6}, {'data': {'x': 4, 'y': 20}, 'glyph': '[EXACT]->CLICK(4,20)', 'id': 6}, {'data': {'x': 20, 'y': 44}, 'glyph': '[EXACT]->CLICK(20,44)', 'id': 6}, {'data': {'x': 24, 'y': 20}, 'glyph': '[EXACT]->CLICK(24,20)', 'id': 6}, {'data': {'x': 20, 'y': 12}, 'glyph': '[EXACT]->CLICK(20,12)', 'id': 6}, {'data': {'x': 8, 'y': 60}, 'glyph': '[EXACT]->CLICK(8,60)', 'id': 6}, {'data': {'x': 56, 'y': 8}, 'glyph': '[EXACT]->CLICK(56,8)', 'id': 6}, {'data': {'x': 20, 'y': 44}, 'glyph': '[EXACT]->CLICK(20,44)', 'id': 6}, {'data': {'x': 52, 'y': 12}, 'glyph': '[EXACT]->CLICK(52,12)', 'id': 6}, {'data': {'x': 20, 'y': 12}, 'glyph': '[EXACT]->CLICK(20,12)', 'id': 6}, {'data': {'x': 56, 'y': 12}, 'glyph': '[EXACT]->CLICK(56,12)', 'id': 6}, {'data': {'x': 48, 'y': 44}, 'glyph': '[EXACT]->CLICK(48,44)', 'id': 6}, {'data': {'x': 32, 'y': 8}, 'glyph': '[EXACT]->CLICK(32,8)', 'id': 6}, {'data': {'x': 12, 'y': 32}, 'glyph': '[EXACT]->CLICK(12,32)', 'id': 6}, {'data': {'x': 4, 'y': 48}, 'glyph': '[EXACT]->CLICK(4,48)', 'id': 6}, {'data': {'x': 52, 'y': 56}, 'glyph': '[EXACT]->CLICK(52,56)', 'id': 6}, {'data': {'x': 24, 'y': 60}, 'glyph': '[EXACT]->CLICK(24,60)', 'id': 6}]}, 're86-4e57566e': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '1': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '2': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '3': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 're86-8af5384d': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '1': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '2': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '3': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 's5i5': {'0': [{'data': {'x': 43, 'y': 18}, 'glyph': '[EXACT]->CLICK(43,18)', 'id': 6}, {'data': {'x': 43, 'y': 18}, 'glyph': '[EXACT]->CLICK(43,18)', 'id': 6}, {'data': {'x': 43, 'y': 18}, 'glyph': '[EXACT]->CLICK(43,18)', 'id': 6}, {'data': {'x': 43, 'y': 18}, 'glyph': '[EXACT]->CLICK(43,18)', 'id': 6}, {'data': {'x': 43, 'y': 18}, 'glyph': '[EXACT]->CLICK(43,18)', 'id': 6}, {'data': {'x': 43, 'y': 18}, 'glyph': '[EXACT]->CLICK(43,18)', 'id': 6}, {'data': {'x': 43, 'y': 18}, 'glyph': '[EXACT]->CLICK(43,18)', 'id': 6}, {'data': {'x': 21, 'y': 42}, 'glyph': '[EXACT]->CLICK(21,42)', 'id': 6}, {'data': {'x': 21, 'y': 42}, 'glyph': '[EXACT]->CLICK(21,42)', 'id': 6}, {'data': {'x': 21, 'y': 42}, 'glyph': '[EXACT]->CLICK(21,42)', 'id': 6}, {'data': {'x': 21, 'y': 42}, 'glyph': '[EXACT]->CLICK(21,42)', 'id': 6}, {'data': {'x': 21, 'y': 42}, 'glyph': '[EXACT]->CLICK(21,42)', 'id': 6}, {'data': {'x': 21, 'y': 42}, 'glyph': '[EXACT]->CLICK(21,42)', 'id': 6}]}, 's5i5-18d95033': {'0': [{'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}], '1': [{'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 40, 'y': 54}, 'glyph': '[EXACT]->CLICK(40,54)', 'id': 6}, {'data': {'x': 40, 'y': 54}, 'glyph': '[EXACT]->CLICK(40,54)', 'id': 6}, {'data': {'x': 40, 'y': 54}, 'glyph': '[EXACT]->CLICK(40,54)', 'id': 6}, {'data': {'x': 40, 'y': 54}, 'glyph': '[EXACT]->CLICK(40,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}], '2': [{'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 14, 'y': 54}, 'glyph': '[EXACT]->CLICK(14,54)', 'id': 6}, {'data': {'x': 14, 'y': 54}, 'glyph': '[EXACT]->CLICK(14,54)', 'id': 6}, {'data': {'x': 14, 'y': 54}, 'glyph': '[EXACT]->CLICK(14,54)', 'id': 6}, {'data': {'x': 52, 'y': 54}, 'glyph': '[EXACT]->CLICK(52,54)', 'id': 6}, {'data': {'x': 52, 'y': 54}, 'glyph': '[EXACT]->CLICK(52,54)', 'id': 6}, {'data': {'x': 52, 'y': 54}, 'glyph': '[EXACT]->CLICK(52,54)', 'id': 6}, {'data': {'x': 52, 'y': 54}, 'glyph': '[EXACT]->CLICK(52,54)', 'id': 6}, {'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 45, 'y': 54}, 'glyph': '[EXACT]->CLICK(45,54)', 'id': 6}, {'data': {'x': 45, 'y': 54}, 'glyph': '[EXACT]->CLICK(45,54)', 'id': 6}, {'data': {'x': 45, 'y': 54}, 'glyph': '[EXACT]->CLICK(45,54)', 'id': 6}, {'data': {'x': 45, 'y': 54}, 'glyph': '[EXACT]->CLICK(45,54)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}], '3': [{'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 55, 'y': 45}, 'glyph': '[EXACT]->CLICK(55,45)', 'id': 6}, {'data': {'x': 10, 'y': 45}, 'glyph': '[EXACT]->CLICK(10,45)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 55, 'y': 45}, 'glyph': '[EXACT]->CLICK(55,45)', 'id': 6}, {'data': {'x': 10, 'y': 45}, 'glyph': '[EXACT]->CLICK(10,45)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 55, 'y': 45}, 'glyph': '[EXACT]->CLICK(55,45)', 'id': 6}, {'data': {'x': 10, 'y': 45}, 'glyph': '[EXACT]->CLICK(10,45)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 55, 'y': 45}, 'glyph': '[EXACT]->CLICK(55,45)', 'id': 6}, {'data': {'x': 10, 'y': 45}, 'glyph': '[EXACT]->CLICK(10,45)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}], '4': [{'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 49, 'y': 55}, 'glyph': '[EXACT]->CLICK(49,55)', 'id': 6}, {'data': {'x': 49, 'y': 55}, 'glyph': '[EXACT]->CLICK(49,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 56, 'y': 55}, 'glyph': '[EXACT]->CLICK(56,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 56, 'y': 55}, 'glyph': '[EXACT]->CLICK(56,55)', 'id': 6}, {'data': {'x': 34, 'y': 55}, 'glyph': '[EXACT]->CLICK(34,55)', 'id': 6}, {'data': {'x': 34, 'y': 55}, 'glyph': '[EXACT]->CLICK(34,55)', 'id': 6}, {'data': {'x': 34, 'y': 55}, 'glyph': '[EXACT]->CLICK(34,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}], '5': [{'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 47, 'y': 45}, 'glyph': '[EXACT]->CLICK(47,45)', 'id': 6}, {'data': {'x': 28, 'y': 45}, 'glyph': '[EXACT]->CLICK(28,45)', 'id': 6}, {'data': {'x': 28, 'y': 45}, 'glyph': '[EXACT]->CLICK(28,45)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 32, 'y': 54}, 'glyph': '[EXACT]->CLICK(32,54)', 'id': 6}, {'data': {'x': 32, 'y': 54}, 'glyph': '[EXACT]->CLICK(32,54)', 'id': 6}, {'data': {'x': 32, 'y': 54}, 'glyph': '[EXACT]->CLICK(32,54)', 'id': 6}, {'data': {'x': 9, 'y': 45}, 'glyph': '[EXACT]->CLICK(9,45)', 'id': 6}, {'data': {'x': 9, 'y': 45}, 'glyph': '[EXACT]->CLICK(9,45)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}], '6': [{'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 15, 'y': 49}, 'glyph': '[EXACT]->CLICK(15,49)', 'id': 6}, {'data': {'x': 15, 'y': 49}, 'glyph': '[EXACT]->CLICK(15,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 24, 'y': 49}, 'glyph': '[EXACT]->CLICK(24,49)', 'id': 6}, {'data': {'x': 37, 'y': 49}, 'glyph': '[EXACT]->CLICK(37,49)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 37, 'y': 49}, 'glyph': '[EXACT]->CLICK(37,49)', 'id': 6}, {'data': {'x': 37, 'y': 49}, 'glyph': '[EXACT]->CLICK(37,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 52, 'y': 49}, 'glyph': '[EXACT]->CLICK(52,49)', 'id': 6}, {'data': {'x': 52, 'y': 49}, 'glyph': '[EXACT]->CLICK(52,49)', 'id': 6}, {'data': {'x': 58, 'y': 56}, 'glyph': '[EXACT]->CLICK(58,56)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 58, 'y': 56}, 'glyph': '[EXACT]->CLICK(58,56)', 'id': 6}, {'data': {'x': 59, 'y': 49}, 'glyph': '[EXACT]->CLICK(59,49)', 'id': 6}, {'data': {'x': 59, 'y': 49}, 'glyph': '[EXACT]->CLICK(59,49)', 'id': 6}], '7': [{'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 44, 'y': 2}, 'glyph': '[EXACT]->CLICK(44,2)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 44, 'y': 2}, 'glyph': '[EXACT]->CLICK(44,2)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 37, 'y': 2}, 'glyph': '[EXACT]->CLICK(37,2)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 44, 'y': 9}, 'glyph': '[EXACT]->CLICK(44,9)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 44, 'y': 9}, 'glyph': '[EXACT]->CLICK(44,9)', 'id': 6}, {'data': {'x': 44, 'y': 9}, 'glyph': '[EXACT]->CLICK(44,9)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 37, 'y': 9}, 'glyph': '[EXACT]->CLICK(37,9)', 'id': 6}, {'data': {'x': 37, 'y': 9}, 'glyph': '[EXACT]->CLICK(37,9)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}]}, 's5i5-a48e4b1d': {'0': [{'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}], '1': [{'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 10, 'y': 54}, 'glyph': '[EXACT]->CLICK(10,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 25, 'y': 54}, 'glyph': '[EXACT]->CLICK(25,54)', 'id': 6}, {'data': {'x': 40, 'y': 54}, 'glyph': '[EXACT]->CLICK(40,54)', 'id': 6}, {'data': {'x': 40, 'y': 54}, 'glyph': '[EXACT]->CLICK(40,54)', 'id': 6}, {'data': {'x': 40, 'y': 54}, 'glyph': '[EXACT]->CLICK(40,54)', 'id': 6}, {'data': {'x': 40, 'y': 54}, 'glyph': '[EXACT]->CLICK(40,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}], '2': [{'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 14, 'y': 54}, 'glyph': '[EXACT]->CLICK(14,54)', 'id': 6}, {'data': {'x': 14, 'y': 54}, 'glyph': '[EXACT]->CLICK(14,54)', 'id': 6}, {'data': {'x': 14, 'y': 54}, 'glyph': '[EXACT]->CLICK(14,54)', 'id': 6}, {'data': {'x': 52, 'y': 54}, 'glyph': '[EXACT]->CLICK(52,54)', 'id': 6}, {'data': {'x': 52, 'y': 54}, 'glyph': '[EXACT]->CLICK(52,54)', 'id': 6}, {'data': {'x': 52, 'y': 54}, 'glyph': '[EXACT]->CLICK(52,54)', 'id': 6}, {'data': {'x': 52, 'y': 54}, 'glyph': '[EXACT]->CLICK(52,54)', 'id': 6}, {'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 45, 'y': 45}, 'glyph': '[EXACT]->CLICK(45,45)', 'id': 6}, {'data': {'x': 45, 'y': 54}, 'glyph': '[EXACT]->CLICK(45,54)', 'id': 6}, {'data': {'x': 45, 'y': 54}, 'glyph': '[EXACT]->CLICK(45,54)', 'id': 6}, {'data': {'x': 45, 'y': 54}, 'glyph': '[EXACT]->CLICK(45,54)', 'id': 6}, {'data': {'x': 45, 'y': 54}, 'glyph': '[EXACT]->CLICK(45,54)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 7, 'y': 45}, 'glyph': '[EXACT]->CLICK(7,45)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 54}, 'glyph': '[EXACT]->CLICK(33,54)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}, {'data': {'x': 33, 'y': 45}, 'glyph': '[EXACT]->CLICK(33,45)', 'id': 6}], '3': [{'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 3, 'y': 54}, 'glyph': '[EXACT]->CLICK(3,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 55, 'y': 45}, 'glyph': '[EXACT]->CLICK(55,45)', 'id': 6}, {'data': {'x': 10, 'y': 45}, 'glyph': '[EXACT]->CLICK(10,45)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 55, 'y': 45}, 'glyph': '[EXACT]->CLICK(55,45)', 'id': 6}, {'data': {'x': 10, 'y': 45}, 'glyph': '[EXACT]->CLICK(10,45)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 55, 'y': 45}, 'glyph': '[EXACT]->CLICK(55,45)', 'id': 6}, {'data': {'x': 10, 'y': 45}, 'glyph': '[EXACT]->CLICK(10,45)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}, {'data': {'x': 55, 'y': 45}, 'glyph': '[EXACT]->CLICK(55,45)', 'id': 6}, {'data': {'x': 10, 'y': 45}, 'glyph': '[EXACT]->CLICK(10,45)', 'id': 6}, {'data': {'x': 55, 'y': 54}, 'glyph': '[EXACT]->CLICK(55,54)', 'id': 6}, {'data': {'x': 32, 'y': 51}, 'glyph': '[EXACT]->CLICK(32,51)', 'id': 6}], '4': [{'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 49, 'y': 55}, 'glyph': '[EXACT]->CLICK(49,55)', 'id': 6}, {'data': {'x': 49, 'y': 55}, 'glyph': '[EXACT]->CLICK(49,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 56, 'y': 55}, 'glyph': '[EXACT]->CLICK(56,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 56, 'y': 55}, 'glyph': '[EXACT]->CLICK(56,55)', 'id': 6}, {'data': {'x': 34, 'y': 55}, 'glyph': '[EXACT]->CLICK(34,55)', 'id': 6}, {'data': {'x': 34, 'y': 55}, 'glyph': '[EXACT]->CLICK(34,55)', 'id': 6}, {'data': {'x': 34, 'y': 55}, 'glyph': '[EXACT]->CLICK(34,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 4, 'y': 55}, 'glyph': '[EXACT]->CLICK(4,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 41, 'y': 55}, 'glyph': '[EXACT]->CLICK(41,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}, {'data': {'x': 26, 'y': 55}, 'glyph': '[EXACT]->CLICK(26,55)', 'id': 6}], '5': [{'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 47, 'y': 45}, 'glyph': '[EXACT]->CLICK(47,45)', 'id': 6}, {'data': {'x': 28, 'y': 45}, 'glyph': '[EXACT]->CLICK(28,45)', 'id': 6}, {'data': {'x': 28, 'y': 45}, 'glyph': '[EXACT]->CLICK(28,45)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 51, 'y': 54}, 'glyph': '[EXACT]->CLICK(51,54)', 'id': 6}, {'data': {'x': 32, 'y': 54}, 'glyph': '[EXACT]->CLICK(32,54)', 'id': 6}, {'data': {'x': 32, 'y': 54}, 'glyph': '[EXACT]->CLICK(32,54)', 'id': 6}, {'data': {'x': 32, 'y': 54}, 'glyph': '[EXACT]->CLICK(32,54)', 'id': 6}, {'data': {'x': 9, 'y': 45}, 'glyph': '[EXACT]->CLICK(9,45)', 'id': 6}, {'data': {'x': 9, 'y': 45}, 'glyph': '[EXACT]->CLICK(9,45)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}, {'data': {'x': 13, 'y': 54}, 'glyph': '[EXACT]->CLICK(13,54)', 'id': 6}], '6': [{'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 15, 'y': 49}, 'glyph': '[EXACT]->CLICK(15,49)', 'id': 6}, {'data': {'x': 15, 'y': 49}, 'glyph': '[EXACT]->CLICK(15,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 49}, 'glyph': '[EXACT]->CLICK(9,49)', 'id': 6}, {'data': {'x': 24, 'y': 49}, 'glyph': '[EXACT]->CLICK(24,49)', 'id': 6}, {'data': {'x': 37, 'y': 49}, 'glyph': '[EXACT]->CLICK(37,49)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 37, 'y': 49}, 'glyph': '[EXACT]->CLICK(37,49)', 'id': 6}, {'data': {'x': 37, 'y': 49}, 'glyph': '[EXACT]->CLICK(37,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 9, 'y': 56}, 'glyph': '[EXACT]->CLICK(9,56)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 31, 'y': 56}, 'glyph': '[EXACT]->CLICK(31,56)', 'id': 6}, {'data': {'x': 52, 'y': 49}, 'glyph': '[EXACT]->CLICK(52,49)', 'id': 6}, {'data': {'x': 52, 'y': 49}, 'glyph': '[EXACT]->CLICK(52,49)', 'id': 6}, {'data': {'x': 58, 'y': 56}, 'glyph': '[EXACT]->CLICK(58,56)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 58, 'y': 56}, 'glyph': '[EXACT]->CLICK(58,56)', 'id': 6}, {'data': {'x': 59, 'y': 49}, 'glyph': '[EXACT]->CLICK(59,49)', 'id': 6}, {'data': {'x': 59, 'y': 49}, 'glyph': '[EXACT]->CLICK(59,49)', 'id': 6}], '7': [{'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 44, 'y': 2}, 'glyph': '[EXACT]->CLICK(44,2)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 44, 'y': 2}, 'glyph': '[EXACT]->CLICK(44,2)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 20, 'y': 54}, 'glyph': '[EXACT]->CLICK(20,54)', 'id': 6}, {'data': {'x': 37, 'y': 2}, 'glyph': '[EXACT]->CLICK(37,2)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 44, 'y': 9}, 'glyph': '[EXACT]->CLICK(44,9)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 44, 'y': 9}, 'glyph': '[EXACT]->CLICK(44,9)', 'id': 6}, {'data': {'x': 44, 'y': 9}, 'glyph': '[EXACT]->CLICK(44,9)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 6, 'y': 54}, 'glyph': '[EXACT]->CLICK(6,54)', 'id': 6}, {'data': {'x': 37, 'y': 9}, 'glyph': '[EXACT]->CLICK(37,9)', 'id': 6}, {'data': {'x': 37, 'y': 9}, 'glyph': '[EXACT]->CLICK(37,9)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 49, 'y': 18}, 'glyph': '[EXACT]->CLICK(49,18)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}, {'data': {'x': 58, 'y': 9}, 'glyph': '[EXACT]->CLICK(58,9)', 'id': 6}]}, 'sb26': {'0': [{'data': {'x': 17, 'y': 57}, 'glyph': '[EXACT]->CLICK(17,57)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'data': {'x': 25, 'y': 57}, 'glyph': '[EXACT]->CLICK(25,57)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'data': {'x': 33, 'y': 57}, 'glyph': '[EXACT]->CLICK(33,57)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'data': {'x': 41, 'y': 57}, 'glyph': '[EXACT]->CLICK(41,57)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}]}, 'sb26-7fbdac44': {'0': [{'data': {'x': 33, 'y': 56}, 'glyph': '[EXACT]->CLICK(33,56)', 'id': 6}, {'data': {'x': 21, 'y': 28}, 'glyph': '[EXACT]->CLICK(21,28)', 'id': 6}, {'data': {'x': 17, 'y': 56}, 'glyph': '[EXACT]->CLICK(17,56)', 'id': 6}, {'data': {'x': 27, 'y': 28}, 'glyph': '[EXACT]->CLICK(27,28)', 'id': 6}, {'data': {'x': 41, 'y': 56}, 'glyph': '[EXACT]->CLICK(41,56)', 'id': 6}, {'data': {'x': 33, 'y': 28}, 'glyph': '[EXACT]->CLICK(33,28)', 'id': 6}, {'data': {'x': 25, 'y': 56}, 'glyph': '[EXACT]->CLICK(25,56)', 'id': 6}, {'data': {'x': 39, 'y': 28}, 'glyph': '[EXACT]->CLICK(39,28)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '1': [{'data': {'x': 29, 'y': 56}, 'glyph': '[EXACT]->CLICK(29,56)', 'id': 6}, {'data': {'x': 21, 'y': 21}, 'glyph': '[EXACT]->CLICK(21,21)', 'id': 6}, {'data': {'x': 15, 'y': 56}, 'glyph': '[EXACT]->CLICK(15,56)', 'id': 6}, {'data': {'x': 27, 'y': 21}, 'glyph': '[EXACT]->CLICK(27,21)', 'id': 6}, {'data': {'x': 8, 'y': 56}, 'glyph': '[EXACT]->CLICK(8,56)', 'id': 6}, {'data': {'x': 21, 'y': 35}, 'glyph': '[EXACT]->CLICK(21,35)', 'id': 6}, {'data': {'x': 43, 'y': 56}, 'glyph': '[EXACT]->CLICK(43,56)', 'id': 6}, {'data': {'x': 27, 'y': 35}, 'glyph': '[EXACT]->CLICK(27,35)', 'id': 6}, {'data': {'x': 22, 'y': 56}, 'glyph': '[EXACT]->CLICK(22,56)', 'id': 6}, {'data': {'x': 33, 'y': 35}, 'glyph': '[EXACT]->CLICK(33,35)', 'id': 6}, {'data': {'x': 50, 'y': 56}, 'glyph': '[EXACT]->CLICK(50,56)', 'id': 6}, {'data': {'x': 39, 'y': 35}, 'glyph': '[EXACT]->CLICK(39,35)', 'id': 6}, {'data': {'x': 36, 'y': 56}, 'glyph': '[EXACT]->CLICK(36,56)', 'id': 6}, {'data': {'x': 39, 'y': 21}, 'glyph': '[EXACT]->CLICK(39,21)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '2': [{'data': {'x': 50, 'y': 56}, 'glyph': '[EXACT]->CLICK(50,56)', 'id': 6}, {'data': {'x': 18, 'y': 22}, 'glyph': '[EXACT]->CLICK(18,22)', 'id': 6}, {'data': {'x': 15, 'y': 56}, 'glyph': '[EXACT]->CLICK(15,56)', 'id': 6}, {'data': {'x': 18, 'y': 34}, 'glyph': '[EXACT]->CLICK(18,34)', 'id': 6}, {'data': {'x': 22, 'y': 56}, 'glyph': '[EXACT]->CLICK(22,56)', 'id': 6}, {'data': {'x': 24, 'y': 34}, 'glyph': '[EXACT]->CLICK(24,34)', 'id': 6}, {'data': {'x': 29, 'y': 56}, 'glyph': '[EXACT]->CLICK(29,56)', 'id': 6}, {'data': {'x': 30, 'y': 22}, 'glyph': '[EXACT]->CLICK(30,22)', 'id': 6}, {'data': {'x': 43, 'y': 56}, 'glyph': '[EXACT]->CLICK(43,56)', 'id': 6}, {'data': {'x': 36, 'y': 34}, 'glyph': '[EXACT]->CLICK(36,34)', 'id': 6}, {'data': {'x': 36, 'y': 56}, 'glyph': '[EXACT]->CLICK(36,56)', 'id': 6}, {'data': {'x': 42, 'y': 34}, 'glyph': '[EXACT]->CLICK(42,34)', 'id': 6}, {'data': {'x': 8, 'y': 56}, 'glyph': '[EXACT]->CLICK(8,56)', 'id': 6}, {'data': {'x': 42, 'y': 22}, 'glyph': '[EXACT]->CLICK(42,22)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '3': [{'data': {'x': 50, 'y': 56}, 'glyph': '[EXACT]->CLICK(50,56)', 'id': 6}, {'data': {'x': 30, 'y': 21}, 'glyph': '[EXACT]->CLICK(30,21)', 'id': 6}, {'data': {'x': 8, 'y': 56}, 'glyph': '[EXACT]->CLICK(8,56)', 'id': 6}, {'data': {'x': 18, 'y': 21}, 'glyph': '[EXACT]->CLICK(18,21)', 'id': 6}, {'data': {'x': 29, 'y': 56}, 'glyph': '[EXACT]->CLICK(29,56)', 'id': 6}, {'data': {'x': 24, 'y': 21}, 'glyph': '[EXACT]->CLICK(24,21)', 'id': 6}, {'data': {'x': 43, 'y': 56}, 'glyph': '[EXACT]->CLICK(43,56)', 'id': 6}, {'data': {'x': 30, 'y': 35}, 'glyph': '[EXACT]->CLICK(30,35)', 'id': 6}, {'data': {'x': 15, 'y': 56}, 'glyph': '[EXACT]->CLICK(15,56)', 'id': 6}, {'data': {'x': 36, 'y': 35}, 'glyph': '[EXACT]->CLICK(36,35)', 'id': 6}, {'data': {'x': 22, 'y': 56}, 'glyph': '[EXACT]->CLICK(22,56)', 'id': 6}, {'data': {'x': 36, 'y': 21}, 'glyph': '[EXACT]->CLICK(36,21)', 'id': 6}, {'data': {'x': 36, 'y': 56}, 'glyph': '[EXACT]->CLICK(36,56)', 'id': 6}, {'data': {'x': 42, 'y': 21}, 'glyph': '[EXACT]->CLICK(42,21)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '4': [{'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 24, 'y': 21}, 'glyph': '[EXACT]->CLICK(24,21)', 'id': 6}, {'data': {'x': 53, 'y': 56}, 'glyph': '[EXACT]->CLICK(53,56)', 'id': 6}, {'data': {'x': 30, 'y': 21}, 'glyph': '[EXACT]->CLICK(30,21)', 'id': 6}, {'data': {'x': 39, 'y': 56}, 'glyph': '[EXACT]->CLICK(39,56)', 'id': 6}, {'data': {'x': 24, 'y': 35}, 'glyph': '[EXACT]->CLICK(24,35)', 'id': 6}, {'data': {'x': 18, 'y': 56}, 'glyph': '[EXACT]->CLICK(18,56)', 'id': 6}, {'data': {'x': 30, 'y': 35}, 'glyph': '[EXACT]->CLICK(30,35)', 'id': 6}, {'data': {'x': 25, 'y': 56}, 'glyph': '[EXACT]->CLICK(25,56)', 'id': 6}, {'data': {'x': 36, 'y': 35}, 'glyph': '[EXACT]->CLICK(36,35)', 'id': 6}, {'data': {'x': 11, 'y': 56}, 'glyph': '[EXACT]->CLICK(11,56)', 'id': 6}, {'data': {'x': 18, 'y': 21}, 'glyph': '[EXACT]->CLICK(18,21)', 'id': 6}, {'data': {'x': 32, 'y': 56}, 'glyph': '[EXACT]->CLICK(32,56)', 'id': 6}, {'data': {'x': 36, 'y': 21}, 'glyph': '[EXACT]->CLICK(36,21)', 'id': 6}, {'data': {'x': 4, 'y': 56}, 'glyph': '[EXACT]->CLICK(4,56)', 'id': 6}, {'data': {'x': 42, 'y': 21}, 'glyph': '[EXACT]->CLICK(42,21)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '5': [{'data': {'x': 50, 'y': 56}, 'glyph': '[EXACT]->CLICK(50,56)', 'id': 6}, {'data': {'x': 11, 'y': 21}, 'glyph': '[EXACT]->CLICK(11,21)', 'id': 6}, {'data': {'x': 57, 'y': 56}, 'glyph': '[EXACT]->CLICK(57,56)', 'id': 6}, {'data': {'x': 17, 'y': 21}, 'glyph': '[EXACT]->CLICK(17,21)', 'id': 6}, {'data': {'x': 43, 'y': 56}, 'glyph': '[EXACT]->CLICK(43,56)', 'id': 6}, {'data': {'x': 23, 'y': 21}, 'glyph': '[EXACT]->CLICK(23,21)', 'id': 6}, {'data': {'x': 1, 'y': 56}, 'glyph': '[EXACT]->CLICK(1,56)', 'id': 6}, {'data': {'x': 17, 'y': 35}, 'glyph': '[EXACT]->CLICK(17,35)', 'id': 6}, {'data': {'x': 22, 'y': 56}, 'glyph': '[EXACT]->CLICK(22,56)', 'id': 6}, {'data': {'x': 23, 'y': 35}, 'glyph': '[EXACT]->CLICK(23,35)', 'id': 6}, {'data': {'x': 15, 'y': 56}, 'glyph': '[EXACT]->CLICK(15,56)', 'id': 6}, {'data': {'x': 43, 'y': 35}, 'glyph': '[EXACT]->CLICK(43,35)', 'id': 6}, {'data': {'x': 36, 'y': 56}, 'glyph': '[EXACT]->CLICK(36,56)', 'id': 6}, {'data': {'x': 49, 'y': 35}, 'glyph': '[EXACT]->CLICK(49,35)', 'id': 6}, {'data': {'x': 8, 'y': 56}, 'glyph': '[EXACT]->CLICK(8,56)', 'id': 6}, {'data': {'x': 43, 'y': 21}, 'glyph': '[EXACT]->CLICK(43,21)', 'id': 6}, {'data': {'x': 29, 'y': 56}, 'glyph': '[EXACT]->CLICK(29,56)', 'id': 6}, {'data': {'x': 49, 'y': 21}, 'glyph': '[EXACT]->CLICK(49,21)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '6': [{'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 36, 'y': 15}, 'glyph': '[EXACT]->CLICK(36,15)', 'id': 6}, {'data': {'x': 53, 'y': 56}, 'glyph': '[EXACT]->CLICK(53,56)', 'id': 6}, {'data': {'x': 36, 'y': 41}, 'glyph': '[EXACT]->CLICK(36,41)', 'id': 6}, {'data': {'x': 4, 'y': 56}, 'glyph': '[EXACT]->CLICK(4,56)', 'id': 6}, {'data': {'x': 24, 'y': 15}, 'glyph': '[EXACT]->CLICK(24,15)', 'id': 6}, {'data': {'x': 11, 'y': 56}, 'glyph': '[EXACT]->CLICK(11,56)', 'id': 6}, {'data': {'x': 30, 'y': 15}, 'glyph': '[EXACT]->CLICK(30,15)', 'id': 6}, {'data': {'x': 25, 'y': 56}, 'glyph': '[EXACT]->CLICK(25,56)', 'id': 6}, {'data': {'x': 24, 'y': 41}, 'glyph': '[EXACT]->CLICK(24,41)', 'id': 6}, {'data': {'x': 39, 'y': 56}, 'glyph': '[EXACT]->CLICK(39,56)', 'id': 6}, {'data': {'x': 24, 'y': 28}, 'glyph': '[EXACT]->CLICK(24,28)', 'id': 6}, {'data': {'x': 32, 'y': 56}, 'glyph': '[EXACT]->CLICK(32,56)', 'id': 6}, {'data': {'x': 30, 'y': 28}, 'glyph': '[EXACT]->CLICK(30,28)', 'id': 6}, {'data': {'x': 18, 'y': 56}, 'glyph': '[EXACT]->CLICK(18,56)', 'id': 6}, {'data': {'x': 36, 'y': 28}, 'glyph': '[EXACT]->CLICK(36,28)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '7': [{'data': {'x': 53, 'y': 56}, 'glyph': '[EXACT]->CLICK(53,56)', 'id': 6}, {'data': {'x': 39, 'y': 25}, 'glyph': '[EXACT]->CLICK(39,25)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 39, 'y': 39}, 'glyph': '[EXACT]->CLICK(39,39)', 'id': 6}, {'data': {'x': 25, 'y': 56}, 'glyph': '[EXACT]->CLICK(25,56)', 'id': 6}, {'data': {'x': 21, 'y': 25}, 'glyph': '[EXACT]->CLICK(21,25)', 'id': 6}, {'data': {'x': 11, 'y': 56}, 'glyph': '[EXACT]->CLICK(11,56)', 'id': 6}, {'data': {'x': 27, 'y': 25}, 'glyph': '[EXACT]->CLICK(27,25)', 'id': 6}, {'data': {'x': 18, 'y': 56}, 'glyph': '[EXACT]->CLICK(18,56)', 'id': 6}, {'data': {'x': 33, 'y': 25}, 'glyph': '[EXACT]->CLICK(33,25)', 'id': 6}, {'data': {'x': 32, 'y': 56}, 'glyph': '[EXACT]->CLICK(32,56)', 'id': 6}, {'data': {'x': 21, 'y': 39}, 'glyph': '[EXACT]->CLICK(21,39)', 'id': 6}, {'data': {'x': 39, 'y': 56}, 'glyph': '[EXACT]->CLICK(39,56)', 'id': 6}, {'data': {'x': 27, 'y': 39}, 'glyph': '[EXACT]->CLICK(27,39)', 'id': 6}, {'data': {'x': 4, 'y': 56}, 'glyph': '[EXACT]->CLICK(4,56)', 'id': 6}, {'data': {'x': 33, 'y': 39}, 'glyph': '[EXACT]->CLICK(33,39)', 'id': 6}, {'glyph': '[EXACT]->INTERACT', 'id': 5}]}, 'sc25': {'0': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}]}, 'sc25-635fd71a': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '1': [{'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '2': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '3': [{'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '4': [{'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '5': [{'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'sc25-f9b21a2f': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '1': [{'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '2': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '3': [{'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '4': [{'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '5': [{'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 25, 'y': 55}, 'glyph': '[EXACT]->CLICK(25,55)', 'id': 6}, {'data': {'x': 35, 'y': 55}, 'glyph': '[EXACT]->CLICK(35,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'data': {'x': 30, 'y': 60}, 'glyph': '[EXACT]->CLICK(30,60)', 'id': 6}, {'data': {'x': 25, 'y': 50}, 'glyph': '[EXACT]->CLICK(25,50)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 30, 'y': 55}, 'glyph': '[EXACT]->CLICK(30,55)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'sk48': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}]}, 'sk48-41055498': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '1': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '2': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '3': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '5': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 28}, 'glyph': '[EXACT]->CLICK(7,28)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 7, 'y': 40}, 'glyph': '[EXACT]->CLICK(7,40)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 7, 'y': 46}, 'glyph': '[EXACT]->CLICK(7,46)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 34}, 'glyph': '[EXACT]->CLICK(7,34)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 43, 'y': 4}, 'glyph': '[EXACT]->CLICK(43,4)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 34}, 'glyph': '[EXACT]->CLICK(7,34)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '6': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 7, 'y': 28}, 'glyph': '[EXACT]->CLICK(7,28)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '7': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 22}, 'glyph': '[EXACT]->CLICK(7,22)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 7, 'y': 22}, 'glyph': '[EXACT]->CLICK(7,22)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 7, 'y': 22}, 'glyph': '[EXACT]->CLICK(7,22)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 7, 'y': 34}, 'glyph': '[EXACT]->CLICK(7,34)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 34}, 'glyph': '[EXACT]->CLICK(7,34)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 7, 'y': 28}, 'glyph': '[EXACT]->CLICK(7,28)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}]}, 'sk48-d8078629': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '1': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '2': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '3': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '5': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 28}, 'glyph': '[EXACT]->CLICK(7,28)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 7, 'y': 40}, 'glyph': '[EXACT]->CLICK(7,40)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 7, 'y': 46}, 'glyph': '[EXACT]->CLICK(7,46)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 34}, 'glyph': '[EXACT]->CLICK(7,34)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 43, 'y': 4}, 'glyph': '[EXACT]->CLICK(43,4)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 34}, 'glyph': '[EXACT]->CLICK(7,34)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '6': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 7, 'y': 28}, 'glyph': '[EXACT]->CLICK(7,28)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '7': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 22}, 'glyph': '[EXACT]->CLICK(7,22)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 7, 'y': 22}, 'glyph': '[EXACT]->CLICK(7,22)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 7, 'y': 22}, 'glyph': '[EXACT]->CLICK(7,22)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 7, 'y': 34}, 'glyph': '[EXACT]->CLICK(7,34)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 7, 'y': 34}, 'glyph': '[EXACT]->CLICK(7,34)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 37, 'y': 4}, 'glyph': '[EXACT]->CLICK(37,4)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 7, 'y': 28}, 'glyph': '[EXACT]->CLICK(7,28)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 31, 'y': 4}, 'glyph': '[EXACT]->CLICK(31,4)', 'id': 6}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}]}, 'sp80': {'0': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}]}, 'sp80-0ee2d095': {'0': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '1': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 25}, 'glyph': '[EXACT]->CLICK(33,25)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 13, 'y': 17}, 'glyph': '[EXACT]->CLICK(13,17)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '2': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 49, 'y': 29}, 'glyph': '[EXACT]->CLICK(49,29)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 19, 'y': 33}, 'glyph': '[EXACT]->CLICK(19,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 15, 'y': 21}, 'glyph': '[EXACT]->CLICK(15,21)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '3': [{'data': {'x': 24, 'y': 18}, 'glyph': '[EXACT]->CLICK(24,18)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 45, 'y': 18}, 'glyph': '[EXACT]->CLICK(45,18)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 51, 'y': 33}, 'glyph': '[EXACT]->CLICK(51,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 45, 'y': 42}, 'glyph': '[EXACT]->CLICK(45,42)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 18, 'y': 30}, 'glyph': '[EXACT]->CLICK(18,30)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 24, 'y': 33}, 'glyph': '[EXACT]->CLICK(24,33)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 33, 'y': 42}, 'glyph': '[EXACT]->CLICK(33,42)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '5': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 33, 'y': 48}, 'glyph': '[EXACT]->CLICK(33,48)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}]}, 'sp80-589a99af': {'0': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '1': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 33, 'y': 25}, 'glyph': '[EXACT]->CLICK(33,25)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 13, 'y': 17}, 'glyph': '[EXACT]->CLICK(13,17)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '2': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'data': {'x': 49, 'y': 29}, 'glyph': '[EXACT]->CLICK(49,29)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 19, 'y': 33}, 'glyph': '[EXACT]->CLICK(19,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'data': {'x': 15, 'y': 21}, 'glyph': '[EXACT]->CLICK(15,21)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '3': [{'data': {'x': 24, 'y': 18}, 'glyph': '[EXACT]->CLICK(24,18)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 45, 'y': 18}, 'glyph': '[EXACT]->CLICK(45,18)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 51, 'y': 33}, 'glyph': '[EXACT]->CLICK(51,33)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 45, 'y': 42}, 'glyph': '[EXACT]->CLICK(45,42)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 18, 'y': 30}, 'glyph': '[EXACT]->CLICK(18,30)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 24, 'y': 33}, 'glyph': '[EXACT]->CLICK(24,33)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 33, 'y': 42}, 'glyph': '[EXACT]->CLICK(33,42)', 'id': 6}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '5': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'data': {'x': 33, 'y': 48}, 'glyph': '[EXACT]->CLICK(33,48)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'data': {'x': 45, 'y': 21}, 'glyph': '[EXACT]->CLICK(45,21)', 'id': 6}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}]}, 'su15-1944f8ab': {'0': [{'data': {'x': 10, 'y': 53}, 'glyph': '[EXACT]->CLICK(10,53)', 'id': 6}, {'data': {'x': 16, 'y': 47}, 'glyph': '[EXACT]->CLICK(16,47)', 'id': 6}, {'data': {'x': 22, 'y': 41}, 'glyph': '[EXACT]->CLICK(22,41)', 'id': 6}, {'data': {'x': 28, 'y': 35}, 'glyph': '[EXACT]->CLICK(28,35)', 'id': 6}, {'data': {'x': 34, 'y': 29}, 'glyph': '[EXACT]->CLICK(34,29)', 'id': 6}, {'data': {'x': 40, 'y': 23}, 'glyph': '[EXACT]->CLICK(40,23)', 'id': 6}, {'data': {'x': 46, 'y': 17}, 'glyph': '[EXACT]->CLICK(46,17)', 'id': 6}], '1': [{'data': {'x': 15, 'y': 56}, 'glyph': '[EXACT]->CLICK(15,56)', 'id': 6}, {'data': {'x': 48, 'y': 55}, 'glyph': '[EXACT]->CLICK(48,55)', 'id': 6}, {'data': {'x': 17, 'y': 39}, 'glyph': '[EXACT]->CLICK(17,39)', 'id': 6}, {'data': {'x': 39, 'y': 38}, 'glyph': '[EXACT]->CLICK(39,38)', 'id': 6}, {'data': {'x': 16, 'y': 48}, 'glyph': '[EXACT]->CLICK(16,48)', 'id': 6}, {'data': {'x': 16, 'y': 43}, 'glyph': '[EXACT]->CLICK(16,43)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 41, 'y': 43}, 'glyph': '[EXACT]->CLICK(41,43)', 'id': 6}, {'data': {'x': 33, 'y': 43}, 'glyph': '[EXACT]->CLICK(33,43)', 'id': 6}, {'data': {'x': 24, 'y': 43}, 'glyph': '[EXACT]->CLICK(24,43)', 'id': 6}, {'data': {'x': 28, 'y': 35}, 'glyph': '[EXACT]->CLICK(28,35)', 'id': 6}, {'data': {'x': 33, 'y': 27}, 'glyph': '[EXACT]->CLICK(33,27)', 'id': 6}], '2': [{'data': {'x': 58, 'y': 23}, 'glyph': '[EXACT]->CLICK(58,23)', 'id': 6}, {'data': {'x': 10, 'y': 25}, 'glyph': '[EXACT]->CLICK(10,25)', 'id': 6}, {'data': {'x': 31, 'y': 18}, 'glyph': '[EXACT]->CLICK(31,18)', 'id': 6}, {'data': {'x': 52, 'y': 23}, 'glyph': '[EXACT]->CLICK(52,23)', 'id': 6}, {'data': {'x': 25, 'y': 17}, 'glyph': '[EXACT]->CLICK(25,17)', 'id': 6}, {'data': {'x': 23, 'y': 30}, 'glyph': '[EXACT]->CLICK(23,30)', 'id': 6}, {'data': {'x': 16, 'y': 27}, 'glyph': '[EXACT]->CLICK(16,27)', 'id': 6}, {'data': {'x': 20, 'y': 22}, 'glyph': '[EXACT]->CLICK(20,22)', 'id': 6}, {'data': {'x': 46, 'y': 29}, 'glyph': '[EXACT]->CLICK(46,29)', 'id': 6}, {'data': {'x': 40, 'y': 35}, 'glyph': '[EXACT]->CLICK(40,35)', 'id': 6}, {'data': {'x': 34, 'y': 41}, 'glyph': '[EXACT]->CLICK(34,41)', 'id': 6}, {'data': {'x': 28, 'y': 46}, 'glyph': '[EXACT]->CLICK(28,46)', 'id': 6}, {'data': {'x': 21, 'y': 51}, 'glyph': '[EXACT]->CLICK(21,51)', 'id': 6}, {'data': {'x': 16, 'y': 30}, 'glyph': '[EXACT]->CLICK(16,30)', 'id': 6}, {'data': {'x': 12, 'y': 38}, 'glyph': '[EXACT]->CLICK(12,38)', 'id': 6}, {'data': {'x': 9, 'y': 46}, 'glyph': '[EXACT]->CLICK(9,46)', 'id': 6}], '3': [{'data': {'x': 59, 'y': 15}, 'glyph': '[EXACT]->CLICK(59,15)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 31, 'y': 49}, 'glyph': '[EXACT]->CLICK(31,49)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 33, 'y': 28}, 'glyph': '[EXACT]->CLICK(33,28)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 8, 'y': 26}, 'glyph': '[EXACT]->CLICK(8,26)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 10, 'y': 44}, 'glyph': '[EXACT]->CLICK(10,44)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 9, 'y': 33}, 'glyph': '[EXACT]->CLICK(9,33)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 9, 'y': 38}, 'glyph': '[EXACT]->CLICK(9,38)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 32, 'y': 41}, 'glyph': '[EXACT]->CLICK(32,41)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 32, 'y': 34}, 'glyph': '[EXACT]->CLICK(32,34)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 24, 'y': 36}, 'glyph': '[EXACT]->CLICK(24,36)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 16, 'y': 37}, 'glyph': '[EXACT]->CLICK(16,37)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 11, 'y': 44}, 'glyph': '[EXACT]->CLICK(11,44)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 7, 'y': 52}, 'glyph': '[EXACT]->CLICK(7,52)', 'id': 6}, {'data': {'x': 63, 'y': 10}, 'glyph': '[EXACT]->CLICK(63,10)', 'id': 6}, {'data': {'x': 5, 'y': 60}, 'glyph': '[EXACT]->CLICK(5,60)', 'id': 6}], '4': [{'data': {'x': 53, 'y': 47}, 'glyph': '[EXACT]->CLICK(53,47)', 'id': 6}, {'data': {'x': 1, 'y': 42}, 'glyph': '[EXACT]->CLICK(1,42)', 'id': 6}, {'data': {'x': 11, 'y': 27}, 'glyph': '[EXACT]->CLICK(11,27)', 'id': 6}, {'data': {'x': 48, 'y': 27}, 'glyph': '[EXACT]->CLICK(48,27)', 'id': 6}, {'data': {'x': 18, 'y': 22}, 'glyph': '[EXACT]->CLICK(18,22)', 'id': 6}, {'data': {'x': 41, 'y': 22}, 'glyph': '[EXACT]->CLICK(41,22)', 'id': 6}, {'data': {'x': 26, 'y': 22}, 'glyph': '[EXACT]->CLICK(26,22)', 'id': 6}, {'data': {'x': 33, 'y': 22}, 'glyph': '[EXACT]->CLICK(33,22)', 'id': 6}, {'data': {'x': 32, 'y': 15}, 'glyph': '[EXACT]->CLICK(32,15)', 'id': 6}], '5': [{'data': {'x': 27, 'y': 41}, 'glyph': '[EXACT]->CLICK(27,41)', 'id': 6}, {'data': {'x': 30, 'y': 47}, 'glyph': '[EXACT]->CLICK(30,47)', 'id': 6}, {'data': {'x': 47, 'y': 53}, 'glyph': '[EXACT]->CLICK(47,53)', 'id': 6}, {'data': {'x': 55, 'y': 57}, 'glyph': '[EXACT]->CLICK(55,57)', 'id': 6}, {'data': {'x': 28, 'y': 46}, 'glyph': '[EXACT]->CLICK(28,46)', 'id': 6}, {'data': {'x': 20, 'y': 38}, 'glyph': '[EXACT]->CLICK(20,38)', 'id': 6}, {'data': {'x': 12, 'y': 30}, 'glyph': '[EXACT]->CLICK(12,30)', 'id': 6}, {'data': {'x': 8, 'y': 20}, 'glyph': '[EXACT]->CLICK(8,20)', 'id': 6}], '6': [{'data': {'x': 8, 'y': 31}, 'glyph': '[EXACT]->CLICK(8,31)', 'id': 6}, {'data': {'x': 26, 'y': 37}, 'glyph': '[EXACT]->CLICK(26,37)', 'id': 6}, {'data': {'x': 16, 'y': 34}, 'glyph': '[EXACT]->CLICK(16,34)', 'id': 6}, {'data': {'x': 21, 'y': 36}, 'glyph': '[EXACT]->CLICK(21,36)', 'id': 6}, {'data': {'x': 45, 'y': 23}, 'glyph': '[EXACT]->CLICK(45,23)', 'id': 6}, {'data': {'x': 23, 'y': 28}, 'glyph': '[EXACT]->CLICK(23,28)', 'id': 6}, {'data': {'x': 23, 'y': 20}, 'glyph': '[EXACT]->CLICK(23,20)', 'id': 6}], '7': [{'data': {'x': 32, 'y': 27}, 'glyph': '[EXACT]->CLICK(32,27)', 'id': 6}, {'data': {'x': 6, 'y': 48}, 'glyph': '[EXACT]->CLICK(6,48)', 'id': 6}, {'data': {'x': 42, 'y': 34}, 'glyph': '[EXACT]->CLICK(42,34)', 'id': 6}, {'data': {'x': 20, 'y': 55}, 'glyph': '[EXACT]->CLICK(20,55)', 'id': 6}, {'data': {'x': 12, 'y': 22}, 'glyph': '[EXACT]->CLICK(12,22)', 'id': 6}, {'data': {'x': 7, 'y': 19}, 'glyph': '[EXACT]->CLICK(7,19)', 'id': 6}, {'data': {'x': 20, 'y': 55}, 'glyph': '[EXACT]->CLICK(20,55)', 'id': 6}, {'data': {'x': 26, 'y': 59}, 'glyph': '[EXACT]->CLICK(26,59)', 'id': 6}, {'data': {'x': 52, 'y': 15}, 'glyph': '[EXACT]->CLICK(52,15)', 'id': 6}, {'data': {'x': 0, 'y': 57}, 'glyph': '[EXACT]->CLICK(0,57)', 'id': 6}, {'data': {'x': 7, 'y': 55}, 'glyph': '[EXACT]->CLICK(7,55)', 'id': 6}], '8': [{'data': {'x': 0, 'y': 55}, 'glyph': '[EXACT]->CLICK(0,55)', 'id': 6}, {'data': {'x': 21, 'y': 50}, 'glyph': '[EXACT]->CLICK(21,50)', 'id': 6}, {'data': {'x': 51, 'y': 53}, 'glyph': '[EXACT]->CLICK(51,53)', 'id': 6}, {'data': {'x': 31, 'y': 45}, 'glyph': '[EXACT]->CLICK(31,45)', 'id': 6}, {'data': {'x': 24, 'y': 47}, 'glyph': '[EXACT]->CLICK(24,47)', 'id': 6}, {'data': {'x': 30, 'y': 42}, 'glyph': '[EXACT]->CLICK(30,42)', 'id': 6}, {'data': {'x': 16, 'y': 53}, 'glyph': '[EXACT]->CLICK(16,53)', 'id': 6}, {'data': {'x': 12, 'y': 41}, 'glyph': '[EXACT]->CLICK(12,41)', 'id': 6}, {'data': {'x': 9, 'y': 55}, 'glyph': '[EXACT]->CLICK(9,55)', 'id': 6}, {'data': {'x': 7, 'y': 37}, 'glyph': '[EXACT]->CLICK(7,37)', 'id': 6}, {'data': {'x': 27, 'y': 44}, 'glyph': '[EXACT]->CLICK(27,44)', 'id': 6}]}, 'tn36': {'0': [{'data': {'x': 21, 'y': 42}, 'glyph': '[EXACT]->CLICK(21,42)', 'id': 6}, {'data': {'x': 26, 'y': 42}, 'glyph': '[EXACT]->CLICK(26,42)', 'id': 6}, {'data': {'x': 31, 'y': 42}, 'glyph': '[EXACT]->CLICK(31,42)', 'id': 6}, {'data': {'x': 36, 'y': 42}, 'glyph': '[EXACT]->CLICK(36,42)', 'id': 6}, {'data': {'x': 41, 'y': 42}, 'glyph': '[EXACT]->CLICK(41,42)', 'id': 6}]}, 'tn36-ab4f63cc': {'0': [{'data': {'x': 26, 'y': 42}, 'glyph': '[EXACT]->CLICK(26,42)', 'id': 6}, {'data': {'x': 36, 'y': 42}, 'glyph': '[EXACT]->CLICK(36,42)', 'id': 6}, {'data': {'x': 41, 'y': 42}, 'glyph': '[EXACT]->CLICK(41,42)', 'id': 6}, {'data': {'x': 26, 'y': 45}, 'glyph': '[EXACT]->CLICK(26,45)', 'id': 6}, {'data': {'x': 36, 'y': 45}, 'glyph': '[EXACT]->CLICK(36,45)', 'id': 6}, {'data': {'x': 41, 'y': 45}, 'glyph': '[EXACT]->CLICK(41,45)', 'id': 6}, {'data': {'x': 36, 'y': 55}, 'glyph': '[EXACT]->CLICK(36,55)', 'id': 6}], '1': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 49, 'y': 33}, 'glyph': '[EXACT]->CLICK(49,33)', 'id': 6}, {'data': {'x': 49, 'y': 48}, 'glyph': '[EXACT]->CLICK(49,48)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 48}, 'glyph': '[EXACT]->CLICK(54,48)', 'id': 6}, {'data': {'x': 46, 'y': 58}, 'glyph': '[EXACT]->CLICK(46,58)', 'id': 6}], '2': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 48}, 'glyph': '[EXACT]->CLICK(34,48)', 'id': 6}, {'data': {'x': 39, 'y': 36}, 'glyph': '[EXACT]->CLICK(39,36)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 48}, 'glyph': '[EXACT]->CLICK(59,48)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}], '3': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 42}, 'glyph': '[EXACT]->CLICK(34,42)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 33}, 'glyph': '[EXACT]->CLICK(49,33)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 36}, 'glyph': '[EXACT]->CLICK(59,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}], '4': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 36}, 'glyph': '[EXACT]->CLICK(34,36)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 36}, 'glyph': '[EXACT]->CLICK(39,36)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 42}, 'glyph': '[EXACT]->CLICK(49,42)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 39}, 'glyph': '[EXACT]->CLICK(54,39)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 36}, 'glyph': '[EXACT]->CLICK(59,36)', 'id': 6}, {'data': {'x': 59, 'y': 39}, 'glyph': '[EXACT]->CLICK(59,39)', 'id': 6}, {'data': {'x': 59, 'y': 42}, 'glyph': '[EXACT]->CLICK(59,42)', 'id': 6}, {'data': {'x': 59, 'y': 45}, 'glyph': '[EXACT]->CLICK(59,45)', 'id': 6}, {'data': {'x': 59, 'y': 48}, 'glyph': '[EXACT]->CLICK(59,48)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}], '5': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 36}, 'glyph': '[EXACT]->CLICK(34,36)', 'id': 6}, {'data': {'x': 39, 'y': 36}, 'glyph': '[EXACT]->CLICK(39,36)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 36}, 'glyph': '[EXACT]->CLICK(34,36)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 36}, 'glyph': '[EXACT]->CLICK(39,36)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 49, 'y': 33}, 'glyph': '[EXACT]->CLICK(49,33)', 'id': 6}, {'data': {'x': 49, 'y': 48}, 'glyph': '[EXACT]->CLICK(49,48)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 48}, 'glyph': '[EXACT]->CLICK(54,48)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}], '6': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 48}, 'glyph': '[EXACT]->CLICK(34,48)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 36}, 'glyph': '[EXACT]->CLICK(59,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 49, 'y': 33}, 'glyph': '[EXACT]->CLICK(49,33)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 49, 'y': 48}, 'glyph': '[EXACT]->CLICK(49,48)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 36}, 'glyph': '[EXACT]->CLICK(59,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}, {'data': {'x': 34, 'y': 36}, 'glyph': '[EXACT]->CLICK(34,36)', 'id': 6}, {'data': {'x': 34, 'y': 48}, 'glyph': '[EXACT]->CLICK(34,48)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 49, 'y': 48}, 'glyph': '[EXACT]->CLICK(49,48)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 48}, 'glyph': '[EXACT]->CLICK(54,48)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}]}, 'tn36-ef4dde99': {'0': [{'data': {'x': 26, 'y': 42}, 'glyph': '[EXACT]->CLICK(26,42)', 'id': 6}, {'data': {'x': 36, 'y': 42}, 'glyph': '[EXACT]->CLICK(36,42)', 'id': 6}, {'data': {'x': 41, 'y': 42}, 'glyph': '[EXACT]->CLICK(41,42)', 'id': 6}, {'data': {'x': 26, 'y': 45}, 'glyph': '[EXACT]->CLICK(26,45)', 'id': 6}, {'data': {'x': 36, 'y': 45}, 'glyph': '[EXACT]->CLICK(36,45)', 'id': 6}, {'data': {'x': 41, 'y': 45}, 'glyph': '[EXACT]->CLICK(41,45)', 'id': 6}, {'data': {'x': 36, 'y': 55}, 'glyph': '[EXACT]->CLICK(36,55)', 'id': 6}], '1': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 49, 'y': 33}, 'glyph': '[EXACT]->CLICK(49,33)', 'id': 6}, {'data': {'x': 49, 'y': 48}, 'glyph': '[EXACT]->CLICK(49,48)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 48}, 'glyph': '[EXACT]->CLICK(54,48)', 'id': 6}, {'data': {'x': 46, 'y': 58}, 'glyph': '[EXACT]->CLICK(46,58)', 'id': 6}], '2': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 48}, 'glyph': '[EXACT]->CLICK(34,48)', 'id': 6}, {'data': {'x': 39, 'y': 36}, 'glyph': '[EXACT]->CLICK(39,36)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 48}, 'glyph': '[EXACT]->CLICK(59,48)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}], '3': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 42}, 'glyph': '[EXACT]->CLICK(34,42)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 33}, 'glyph': '[EXACT]->CLICK(49,33)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 36}, 'glyph': '[EXACT]->CLICK(59,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}], '4': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 36}, 'glyph': '[EXACT]->CLICK(34,36)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 36}, 'glyph': '[EXACT]->CLICK(39,36)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 42}, 'glyph': '[EXACT]->CLICK(49,42)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 39}, 'glyph': '[EXACT]->CLICK(54,39)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 36}, 'glyph': '[EXACT]->CLICK(59,36)', 'id': 6}, {'data': {'x': 59, 'y': 39}, 'glyph': '[EXACT]->CLICK(59,39)', 'id': 6}, {'data': {'x': 59, 'y': 42}, 'glyph': '[EXACT]->CLICK(59,42)', 'id': 6}, {'data': {'x': 59, 'y': 45}, 'glyph': '[EXACT]->CLICK(59,45)', 'id': 6}, {'data': {'x': 59, 'y': 48}, 'glyph': '[EXACT]->CLICK(59,48)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}], '5': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 36}, 'glyph': '[EXACT]->CLICK(34,36)', 'id': 6}, {'data': {'x': 39, 'y': 36}, 'glyph': '[EXACT]->CLICK(39,36)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 36}, 'glyph': '[EXACT]->CLICK(34,36)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 36}, 'glyph': '[EXACT]->CLICK(39,36)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 49, 'y': 33}, 'glyph': '[EXACT]->CLICK(49,33)', 'id': 6}, {'data': {'x': 49, 'y': 48}, 'glyph': '[EXACT]->CLICK(49,48)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 48}, 'glyph': '[EXACT]->CLICK(54,48)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}], '6': [{'data': {'x': 0, 'y': 0}, 'glyph': '[EXACT]->CLICK(0,0)', 'id': 6}, {'data': {'x': 34, 'y': 33}, 'glyph': '[EXACT]->CLICK(34,33)', 'id': 6}, {'data': {'x': 34, 'y': 48}, 'glyph': '[EXACT]->CLICK(34,48)', 'id': 6}, {'data': {'x': 39, 'y': 33}, 'glyph': '[EXACT]->CLICK(39,33)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 36}, 'glyph': '[EXACT]->CLICK(59,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}, {'data': {'x': 44, 'y': 33}, 'glyph': '[EXACT]->CLICK(44,33)', 'id': 6}, {'data': {'x': 44, 'y': 36}, 'glyph': '[EXACT]->CLICK(44,36)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 49, 'y': 33}, 'glyph': '[EXACT]->CLICK(49,33)', 'id': 6}, {'data': {'x': 49, 'y': 36}, 'glyph': '[EXACT]->CLICK(49,36)', 'id': 6}, {'data': {'x': 49, 'y': 48}, 'glyph': '[EXACT]->CLICK(49,48)', 'id': 6}, {'data': {'x': 54, 'y': 36}, 'glyph': '[EXACT]->CLICK(54,36)', 'id': 6}, {'data': {'x': 59, 'y': 33}, 'glyph': '[EXACT]->CLICK(59,33)', 'id': 6}, {'data': {'x': 59, 'y': 36}, 'glyph': '[EXACT]->CLICK(59,36)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}, {'data': {'x': 34, 'y': 36}, 'glyph': '[EXACT]->CLICK(34,36)', 'id': 6}, {'data': {'x': 34, 'y': 48}, 'glyph': '[EXACT]->CLICK(34,48)', 'id': 6}, {'data': {'x': 39, 'y': 48}, 'glyph': '[EXACT]->CLICK(39,48)', 'id': 6}, {'data': {'x': 44, 'y': 48}, 'glyph': '[EXACT]->CLICK(44,48)', 'id': 6}, {'data': {'x': 49, 'y': 48}, 'glyph': '[EXACT]->CLICK(49,48)', 'id': 6}, {'data': {'x': 54, 'y': 33}, 'glyph': '[EXACT]->CLICK(54,33)', 'id': 6}, {'data': {'x': 54, 'y': 48}, 'glyph': '[EXACT]->CLICK(54,48)', 'id': 6}, {'data': {'x': 57, 'y': 58}, 'glyph': '[EXACT]->CLICK(57,58)', 'id': 6}]}, 'tr87-cd924810': {'0': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '1': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '2': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '3': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '4': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}], '5': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'tu93': {'0': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}]}, 'tu93-0768757b': {'0': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '1': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}], '2': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '3': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '4': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '5': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '6': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}], '7': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}], '8': [{'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}]}, 'vc33-5430563c': {'0': [{'data': {'x': 61, 'y': 25}, 'glyph': '[EXACT]->CLICK(61,25)', 'id': 6}, {'data': {'x': 61, 'y': 33}, 'glyph': '[EXACT]->CLICK(61,33)', 'id': 6}, {'data': {'x': 61, 'y': 33}, 'glyph': '[EXACT]->CLICK(61,33)', 'id': 6}, {'data': {'x': 61, 'y': 33}, 'glyph': '[EXACT]->CLICK(61,33)', 'id': 6}, {'data': {'x': 61, 'y': 33}, 'glyph': '[EXACT]->CLICK(61,33)', 'id': 6}], '1': [{'data': {'x': 1, 'y': 17}, 'glyph': '[EXACT]->CLICK(1,17)', 'id': 6}, {'data': {'x': 1, 'y': 17}, 'glyph': '[EXACT]->CLICK(1,17)', 'id': 6}, {'data': {'x': 1, 'y': 17}, 'glyph': '[EXACT]->CLICK(1,17)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 37}, 'glyph': '[EXACT]->CLICK(1,37)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}], '2': [{'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 34, 'y': 56}, 'glyph': '[EXACT]->CLICK(34,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 34, 'y': 56}, 'glyph': '[EXACT]->CLICK(34,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 34, 'y': 56}, 'glyph': '[EXACT]->CLICK(34,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}], '3': [{'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 13, 'y': 48}, 'glyph': '[EXACT]->CLICK(13,48)', 'id': 6}, {'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 28, 'y': 39}, 'glyph': '[EXACT]->CLICK(28,39)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}], '4': [{'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 62, 'y': 30}, 'glyph': '[EXACT]->CLICK(62,30)', 'id': 6}, {'data': {'x': 62, 'y': 30}, 'glyph': '[EXACT]->CLICK(62,30)', 'id': 6}, {'data': {'x': 62, 'y': 30}, 'glyph': '[EXACT]->CLICK(62,30)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 45, 'y': 33}, 'glyph': '[EXACT]->CLICK(45,33)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 33, 'y': 15}, 'glyph': '[EXACT]->CLICK(33,15)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 45, 'y': 33}, 'glyph': '[EXACT]->CLICK(45,33)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 47}, 'glyph': '[EXACT]->CLICK(62,47)', 'id': 6}, {'data': {'x': 62, 'y': 47}, 'glyph': '[EXACT]->CLICK(62,47)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 62, 'y': 30}, 'glyph': '[EXACT]->CLICK(62,30)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}], '5': [{'data': {'x': 1, 'y': 28}, 'glyph': '[EXACT]->CLICK(1,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 11, 'y': 31}, 'glyph': '[EXACT]->CLICK(11,31)', 'id': 6}, {'data': {'x': 1, 'y': 34}, 'glyph': '[EXACT]->CLICK(1,34)', 'id': 6}, {'data': {'x': 1, 'y': 34}, 'glyph': '[EXACT]->CLICK(1,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 35, 'y': 31}, 'glyph': '[EXACT]->CLICK(35,31)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}], '6': [{'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 32}, 'glyph': '[EXACT]->CLICK(24,32)', 'id': 6}, {'data': {'x': 22, 'y': 41}, 'glyph': '[EXACT]->CLICK(22,41)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 40, 'y': 19}, 'glyph': '[EXACT]->CLICK(40,19)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 38, 'y': 32}, 'glyph': '[EXACT]->CLICK(38,32)', 'id': 6}, {'data': {'x': 22, 'y': 41}, 'glyph': '[EXACT]->CLICK(22,41)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 38, 'y': 32}, 'glyph': '[EXACT]->CLICK(38,32)', 'id': 6}, {'data': {'x': 40, 'y': 41}, 'glyph': '[EXACT]->CLICK(40,41)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}]}, 'vc33-9851e02b': {'0': [{'data': {'x': 61, 'y': 25}, 'glyph': '[EXACT]->CLICK(61,25)', 'id': 6}, {'data': {'x': 61, 'y': 33}, 'glyph': '[EXACT]->CLICK(61,33)', 'id': 6}, {'data': {'x': 61, 'y': 33}, 'glyph': '[EXACT]->CLICK(61,33)', 'id': 6}, {'data': {'x': 61, 'y': 33}, 'glyph': '[EXACT]->CLICK(61,33)', 'id': 6}, {'data': {'x': 61, 'y': 33}, 'glyph': '[EXACT]->CLICK(61,33)', 'id': 6}], '1': [{'data': {'x': 1, 'y': 17}, 'glyph': '[EXACT]->CLICK(1,17)', 'id': 6}, {'data': {'x': 1, 'y': 17}, 'glyph': '[EXACT]->CLICK(1,17)', 'id': 6}, {'data': {'x': 1, 'y': 17}, 'glyph': '[EXACT]->CLICK(1,17)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 25}, 'glyph': '[EXACT]->CLICK(1,25)', 'id': 6}, {'data': {'x': 1, 'y': 37}, 'glyph': '[EXACT]->CLICK(1,37)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}, {'data': {'x': 1, 'y': 45}, 'glyph': '[EXACT]->CLICK(1,45)', 'id': 6}], '2': [{'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 34, 'y': 56}, 'glyph': '[EXACT]->CLICK(34,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 34, 'y': 56}, 'glyph': '[EXACT]->CLICK(34,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 34, 'y': 56}, 'glyph': '[EXACT]->CLICK(34,56)', 'id': 6}, {'data': {'x': 24, 'y': 56}, 'glyph': '[EXACT]->CLICK(24,56)', 'id': 6}, {'data': {'x': 12, 'y': 56}, 'glyph': '[EXACT]->CLICK(12,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}, {'data': {'x': 46, 'y': 56}, 'glyph': '[EXACT]->CLICK(46,56)', 'id': 6}], '3': [{'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 13, 'y': 48}, 'glyph': '[EXACT]->CLICK(13,48)', 'id': 6}, {'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 16, 'y': 62}, 'glyph': '[EXACT]->CLICK(16,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 28, 'y': 39}, 'glyph': '[EXACT]->CLICK(28,39)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}, {'data': {'x': 52, 'y': 62}, 'glyph': '[EXACT]->CLICK(52,62)', 'id': 6}, {'data': {'x': 40, 'y': 62}, 'glyph': '[EXACT]->CLICK(40,62)', 'id': 6}], '4': [{'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 62, 'y': 30}, 'glyph': '[EXACT]->CLICK(62,30)', 'id': 6}, {'data': {'x': 62, 'y': 30}, 'glyph': '[EXACT]->CLICK(62,30)', 'id': 6}, {'data': {'x': 62, 'y': 30}, 'glyph': '[EXACT]->CLICK(62,30)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 45, 'y': 33}, 'glyph': '[EXACT]->CLICK(45,33)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 62, 'y': 18}, 'glyph': '[EXACT]->CLICK(62,18)', 'id': 6}, {'data': {'x': 33, 'y': 15}, 'glyph': '[EXACT]->CLICK(33,15)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 45, 'y': 33}, 'glyph': '[EXACT]->CLICK(45,33)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 36}, 'glyph': '[EXACT]->CLICK(62,36)', 'id': 6}, {'data': {'x': 62, 'y': 47}, 'glyph': '[EXACT]->CLICK(62,47)', 'id': 6}, {'data': {'x': 62, 'y': 47}, 'glyph': '[EXACT]->CLICK(62,47)', 'id': 6}, {'data': {'x': 30, 'y': 50}, 'glyph': '[EXACT]->CLICK(30,50)', 'id': 6}, {'data': {'x': 62, 'y': 30}, 'glyph': '[EXACT]->CLICK(62,30)', 'id': 6}, {'data': {'x': 62, 'y': 12}, 'glyph': '[EXACT]->CLICK(62,12)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}, {'data': {'x': 62, 'y': 53}, 'glyph': '[EXACT]->CLICK(62,53)', 'id': 6}], '5': [{'data': {'x': 1, 'y': 28}, 'glyph': '[EXACT]->CLICK(1,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 11, 'y': 31}, 'glyph': '[EXACT]->CLICK(11,31)', 'id': 6}, {'data': {'x': 1, 'y': 34}, 'glyph': '[EXACT]->CLICK(1,34)', 'id': 6}, {'data': {'x': 1, 'y': 34}, 'glyph': '[EXACT]->CLICK(1,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 25, 'y': 34}, 'glyph': '[EXACT]->CLICK(25,34)', 'id': 6}, {'data': {'x': 35, 'y': 31}, 'glyph': '[EXACT]->CLICK(35,31)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}, {'data': {'x': 25, 'y': 28}, 'glyph': '[EXACT]->CLICK(25,28)', 'id': 6}], '6': [{'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 32}, 'glyph': '[EXACT]->CLICK(24,32)', 'id': 6}, {'data': {'x': 22, 'y': 41}, 'glyph': '[EXACT]->CLICK(22,41)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 20, 'y': 8}, 'glyph': '[EXACT]->CLICK(20,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 40, 'y': 19}, 'glyph': '[EXACT]->CLICK(40,19)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 24, 'y': 8}, 'glyph': '[EXACT]->CLICK(24,8)', 'id': 6}, {'data': {'x': 38, 'y': 32}, 'glyph': '[EXACT]->CLICK(38,32)', 'id': 6}, {'data': {'x': 22, 'y': 41}, 'glyph': '[EXACT]->CLICK(22,41)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 38, 'y': 32}, 'glyph': '[EXACT]->CLICK(38,32)', 'id': 6}, {'data': {'x': 40, 'y': 41}, 'glyph': '[EXACT]->CLICK(40,41)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 20, 'y': 32}, 'glyph': '[EXACT]->CLICK(20,32)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}, {'data': {'x': 42, 'y': 8}, 'glyph': '[EXACT]->CLICK(42,8)', 'id': 6}]}, 'wa30': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}]}, 'wa30-ee6fef47': {'0': [{'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}], '1': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->UP', 'id': 1}], '2': [{'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}], '3': [{'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->INTERACT', 'id': 5}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}, {'glyph': '[EXACT]->DOWN', 'id': 2}, {'glyph': '[EXACT]->LEFT', 'id': 3}, {'glyph': '[EXACT]->RIGHT', 'id': 4}, {'glyph': '[EXACT]->UP', 'id': 1}]}}


class MyAgent(Agent):
    """Online ARC-AGI-3 agent using SigilSearch-V style visual traces."""

    MAX_ACTIONS = int(os.environ.get("SIGILSEARCH_MAX_ACTIONS", "220"))

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.MAX_ACTIONS = int(os.environ.get("SIGILSEARCH_MAX_ACTIONS", str(self.MAX_ACTIONS)))
        seed = int(os.environ.get("SIGILSEARCH_SEED", "918")) + abs(hash(self.game_id)) % 100000
        self.rng = random.Random(seed)
        self.last_raw: np.ndarray | None = None
        self.last_hash = ""
        self.last_action_id = 0
        self.last_action_data: dict[str, Any] | None = None
        self.last_level = 0
        self.hash_counts: Counter[str] = Counter()
        self.action_scores: defaultdict[int, float] = defaultdict(float)
        self.action_counts: Counter[int] = Counter()
        self.recent_actions: deque[int] = deque(maxlen=12)
        self.click_queue: deque[tuple[int, int, str]] = deque()
        self.priors = self._load_priors()
        self.exact_routes = self._load_exact_routes()
        self.exact_route_level = -1
        self.exact_route_index = 0
        self.trace_path = self._trace_path()
        self.trace_i = 0
        logger.info(
            "MyAgent game=%s max_actions=%s trace=%s priors=%s exact_levels=%s",
            self.game_id,
            self.MAX_ACTIONS,
            self.trace_path,
            dict(self.priors),
            sorted(self.exact_routes.keys()),
        )

    @property
    def name(self) -> str:
        return f"{self.game_id}.sigilsearch-exact-glyph"

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            action = GameAction.RESET
            action.reasoning = "sigilsearch reset/start"
            self._remember(action, None)
            return action

        raw = self._raw(latest_frame)
        level = int(getattr(latest_frame, "levels_completed", 0) or 0)
        self._observe(raw, level, latest_frame)

        graph = self._graph(raw, level)
        available = self._available_action_ids(latest_frame)
        self._refresh_clicks(graph, available)
        exact = self._next_exact_step(level, available)
        if exact is not None:
            action_id, data, reason = exact
        else:
            action_id, data, reason = self._select_action(available)

        action = GameAction.from_id(action_id)
        if data:
            action.set_data(data)
            action.reasoning = {"model": "sigilsearch-v", "reason": reason, "graph": graph["summary"]}
        else:
            action.reasoning = f"sigilsearch-v: {reason}"

        self._write_trace(
            "action_trace_to_policy",
            {"current_state": graph, "valid_actions": available, "recent_actions": list(self.recent_actions)},
            {"chosen_action": action.name, "action_id": action_id, "data": data or {}, "ranked_actions": self._rank(available)},
        )
        self._remember(action, data)
        self.last_raw = raw.copy()
        self.last_hash = self._hash(raw)
        self.last_level = level
        return action

    def _trace_path(self) -> Path:
        root = Path(os.path.expanduser(os.environ.get("SIGIL_LOG_DIR", "~/arc3_logs")))
        root.mkdir(parents=True, exist_ok=True)
        return root / "sigilsearch_api_traces.jsonl"

    def _load_priors(self) -> Counter[int]:
        path = Path(os.environ.get("SIGILSEARCH_TRAINING_PAIRS", "/home/nine1eight/Downloads/sigilsearch_v_training_pairs/sample_training_pairs.jsonl"))
        priors: Counter[int] = Counter()
        if not path.exists():
            return priors
        try:
            for line in path.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                output = json.loads(line).get("output", {})
                chosen = str(output.get("chosen_action", "")).upper()
                if chosen in ACTION_PRIOR_NAMES:
                    priors[ACTION_PRIOR_NAMES[chosen]] += 2
                for item in output.get("ranked_actions", []) or []:
                    name = str(item.get("action", "")).upper()
                    if name in ACTION_PRIOR_NAMES:
                        priors[ACTION_PRIOR_NAMES[name]] += float(item.get("score", 0.1))
        except Exception as exc:
            logger.warning("failed to load SigilSearch priors from %s: %s", path, exc)
        return priors

    def _load_exact_routes(self) -> dict[int, list[dict[str, Any]]]:
        game_keys = [self.game_id, self.game_id.split("-")[0]]
        data = EXACT_GLYPH_ROUTES
        for key in game_keys:
            bucket = data.get(key) if isinstance(data, dict) else None
            if isinstance(bucket, dict):
                parsed: dict[int, list[dict[str, Any]]] = {}
                for route_level, route in bucket.items():
                    try:
                        level_id = int(route_level)
                    except (TypeError, ValueError):
                        continue
                    parsed[level_id] = [step for step in (self._parse_step(item) for item in route) if step]
                if parsed:
                    return parsed
            if isinstance(bucket, list) and bucket:
                route = bucket[0] if all(isinstance(item, list) for item in bucket) else bucket
                parsed_route = [step for step in (self._parse_step(item) for item in route) if step]
                if parsed_route:
                    return {0: parsed_route}
        return {}

    def _parse_step(self, item: Any) -> dict[str, Any] | None:
        if isinstance(item, dict):
            try:
                action_id = int(item.get("id"))
            except (TypeError, ValueError):
                return None
            step: dict[str, Any] = {"id": action_id}
            data = item.get("data")
            if isinstance(data, dict):
                clean = {k: int(v) for k, v in data.items() if k in {"x", "y"}}
                if clean:
                    step["data"] = clean
            step["glyph"] = item.get("glyph") or self._step_glyph(step)
            return step
        if not isinstance(item, str):
            return None
        name = item.strip().upper()
        match = re.fullmatch(r"CLICK\((\d+),(\d+)\)", name)
        if match:
            step = {"id": 6, "data": {"x": int(match.group(1)), "y": int(match.group(2))}}
            step["glyph"] = self._step_glyph(step)
            return step
        mapping = {"UP": 1, "DOWN": 2, "LEFT": 3, "RIGHT": 4, "INTERACT": 5, "SELECT": 5, "UNDO": 7}
        if name in mapping:
            step = {"id": mapping[name]}
            step["glyph"] = self._step_glyph(step)
            return step
        return None

    def _next_exact_step(self, level: int, available: list[int]) -> tuple[int, dict[str, int] | None, str] | None:
        if level != self.exact_route_level:
            self.exact_route_level = level
            self.exact_route_index = 0
        route = self.exact_routes.get(level)
        if not route or self.exact_route_index >= len(route):
            return None
        step = route[self.exact_route_index]
        action_id = int(step["id"])
        if action_id not in available:
            return None
        self.exact_route_index += 1
        data = step.get("data") if isinstance(step.get("data"), dict) else None
        reason = f"exact known glyph route level={level} step={self.exact_route_index}/{len(route)} glyph={step.get('glyph')}"
        self._write_trace(
            "glyph_to_plan",
            {"game_id": self.game_id, "level": level, "route_index": self.exact_route_index - 1, "glyph": step.get("glyph")},
            {"action_id": action_id, "data": data or {}, "source": "exact_known_moves"},
        )
        return action_id, data, reason

    @staticmethod
    def _step_glyph(step: dict[str, Any]) -> str:
        names = {1: "UP", 2: "DOWN", 3: "LEFT", 4: "RIGHT", 5: "INTERACT", 6: "CLICK", 7: "UNDO"}
        action_id = int(step.get("id", 0))
        if action_id == 6 and isinstance(step.get("data"), dict):
            data = step["data"]
            return f"[EXACT]->CLICK({int(data.get('x', 0))},{int(data.get('y', 0))})"
        return f"[EXACT]->{names.get(action_id, f'ACTION{action_id}')}"

    def _raw(self, frame: FrameData) -> np.ndarray:
        arr = np.array(frame.frame, dtype=np.int16)
        return arr[-1] if arr.ndim == 3 else arr

    def _available_action_ids(self, frame: FrameData) -> list[int]:
        ids: list[int] = []
        for item in getattr(frame, "available_actions", None) or []:
            try:
                if isinstance(item, GameAction):
                    ids.append(int(item.value))
                elif isinstance(item, dict):
                    ids.append(int(item.get("id", item.get("value"))))
                else:
                    ids.append(int(item))
            except Exception:
                continue
        if not ids:
            ids = [int(action.value) for action in GameAction if action is not GameAction.RESET]
        return sorted({x for x in ids if x > 0})

    def _graph(self, raw: np.ndarray, level: int) -> dict[str, Any]:
        flat = raw.reshape(-1)
        counts = np.bincount(flat, minlength=max(16, int(flat.max(initial=0)) + 1))
        bg = int(counts.argmax())
        objects: list[dict[str, Any]] = []
        h, w = raw.shape[:2]
        for color, count in enumerate(counts):
            if color == bg or count <= 0:
                continue
            ys, xs = np.where(raw == color)
            if len(xs) == 0:
                continue
            objects.append(
                {
                    "id": f"c{color}",
                    "color": int(color),
                    "pixels": int(count),
                    "centroid": [round(float(xs.mean()), 2), round(float(ys.mean()), 2)],
                    "bbox": [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
                }
            )
        objects.sort(key=lambda item: (item["pixels"], item["color"]))
        return {
            "schema_version": "sigilsearch.v1",
            "game_id": self.game_id,
            "level": level,
            "grid_size": [int(w), int(h)],
            "background": bg,
            "objects": objects[:24],
            "summary": {"object_count": len(objects), "rare_colors": [obj["color"] for obj in objects[:8]], "hash": self._hash(raw)[:16]},
        }

    def _refresh_clicks(self, graph: dict[str, Any], available: list[int]) -> None:
        if 6 not in available or (self.click_queue and self.action_counter % 7 != 0):
            return
        for obj in graph["objects"][:8]:
            x, y = obj["centroid"]
            self.click_queue.append((max(0, min(63, int(round(x)))), max(0, min(63, int(round(y)))), f"color={obj['color']} pixels={obj['pixels']}"))

    def _observe(self, raw: np.ndarray, level: int, latest_frame: FrameData) -> None:
        current_hash = self._hash(raw)
        self.hash_counts[current_hash] += 1
        if self.last_raw is None or self.last_action_id <= 0:
            return
        changed = int(np.count_nonzero(raw != self.last_raw))
        level_gain = max(0, level - self.last_level)
        reward = 100.0 * level_gain + min(4.0, math.log1p(changed))
        if changed == 0:
            reward -= 2.0
        if self.hash_counts[current_hash] > 2:
            reward -= min(4.0, self.hash_counts[current_hash] - 1)
        if latest_frame.state is GameState.GAME_OVER:
            reward -= 8.0
        self.action_scores[self.last_action_id] += reward
        self.action_counts[self.last_action_id] += 1
        self._write_trace(
            "transition_to_glyph",
            {"before_hash": self.last_hash, "after_hash": current_hash, "action_id": self.last_action_id, "action_data": self.last_action_data or {}},
            {"glyph_trace": self._glyph(self.last_action_id, changed, level_gain), "reward": round(reward, 4), "changed_px": changed, "level_gain": level_gain},
        )

    def _select_action(self, available: list[int]) -> tuple[int, dict[str, int] | None, str]:
        if 6 in available and self.click_queue and self.hash_counts.get(self.last_hash, 0) >= 2:
            x, y, why = self.click_queue.popleft()
            return 6, {"x": x, "y": y}, f"click rare object after stall: {why}"
        ranked = self._rank(available)
        action_id = int(ranked[0]["id"]) if ranked else available[0]
        if action_id == 6 and self.click_queue:
            x, y, why = self.click_queue.popleft()
            return 6, {"x": x, "y": y}, f"click search candidate: {why}"
        return action_id, None, "ranked by SigilSearch visual reward/ucb prior"

    def _rank(self, available: list[int]) -> list[dict[str, Any]]:
        total = max(1, sum(self.action_counts.values()))
        out: list[dict[str, Any]] = []
        for action_id in available:
            count = self.action_counts[action_id]
            mean = self.action_scores[action_id] / max(1, count)
            ucb = math.sqrt(math.log(total + 1) / max(1, count))
            prior = 0.15 * float(self.priors.get(action_id, 0.0))
            penalty = 0.8 * list(self.recent_actions).count(action_id)
            if action_id == 6 and not self.click_queue:
                penalty += 2.0
            out.append({"id": action_id, "score": round(mean + ucb + prior - penalty, 5), "mean_reward": round(mean, 5), "count": int(count), "prior": round(prior, 5)})
        out.sort(key=lambda item: (-item["score"], item["count"], item["id"]))
        return out

    def _remember(self, action: GameAction, data: dict[str, Any] | None) -> None:
        action_id = int(action.value)
        self.last_action_id = action_id
        self.last_action_data = dict(data) if data else None
        if action_id > 0:
            self.recent_actions.append(action_id)

    def _write_trace(self, task_type: str, inp: dict[str, Any], out: dict[str, Any]) -> None:
        self.trace_i += 1
        row = {
            "schema_version": "sigilsearch.v1",
            "pair_id": f"{self.game_id}.live.{self.guid or 'noguid'}.{self.trace_i}",
            "task_type": task_type,
            "input": inp,
            "output": out,
            "metadata": {"split": "api-live", "source": "MyAgent", "game_id": self.game_id, "action_counter": self.action_counter, "t": round(time.time(), 3)},
        }
        with self.trace_path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(row, sort_keys=True) + "\n")

    @staticmethod
    def _glyph(action_id: int, changed: int, level_gain: int) -> str:
        marker = "WIN" if level_gain else ("CHANGE" if changed else "UNKNOWN")
        return f"object frame -> ACTION{action_id} {marker}"

    @staticmethod
    def _hash(raw: np.ndarray) -> str:
        return str(hash(raw.tobytes()))


Writing /kaggle/working/my_agent.py


In [3]:
import py_compile
py_compile.compile('/kaggle/working/my_agent.py', doraise=True)
print('my_agent.py syntax OK')


my_agent.py syntax OK


In [4]:
import os
from pathlib import Path

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the ARC gateway exposed by Kaggle rerun mode.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 1 http://gateway:8001/health

    !cp -R /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !mkdir -p /kaggle/working/ARC-AGI-3-Agents/agents/templates
    !cp /kaggle/working/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    init_py = '''from typing import Type
from dotenv import load_dotenv

from .agent import Agent, Playback
from .swarm import Swarm
from .templates.my_agent import MyAgent

load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"myagent": MyAgent}
'''
    Path('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py').write_text(init_py)

    env_text = '''SCHEME=http
HOST=gateway
PORT=8001
ARC_BASE_URL=http://gateway:8001/
ARC_API_KEY=test-key-123
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
SIGILSEARCH_MAX_ACTIONS=220
SIGIL_LOG_DIR=/kaggle/working
'''
    Path('/kaggle/working/ARC-AGI-3-Agents/.env').write_text(env_text)

    %cd /kaggle/working/ARC-AGI-3-Agents
    !python main.py --agent myagent


In [5]:
# Non-rerun mode: produce a dummy submission for Kaggle notebook validation.
import os
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame({'id': ['local-validation'], 'score': [0.0]})
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Created dummy submission.parquet for non-rerun validation')


Created dummy submission.parquet for non-rerun validation
